In [9]:
import pandas as pd
import numpy as np
import matplotlib as mpl 
import pybaseball as pb
from pybaseball import statcast
import sklearn
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, mean_squared_error
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from datetime import date
import requests
from bs4 import BeautifulSoup
import duckdb

In [10]:
player_dim = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/player_dim_live.csv')

In [11]:
# Function web scrapes daily lineup information from Rotowire

def get_daily_lineups1():
    url = 'https://www.rotowire.com/baseball/daily-lineups.php'
    response = requests.get(url)
    soup = BeautifulSoup(response.content, 'html.parser')
    
    def extract_team_data(team_list, team_abbr):
        lineup_data = []
        for player in team_list.select('.lineup__player'):
            pos = player.find('div', class_ = 'lineup__pos').text.strip()
            name = player.find('a').get('title', player.find('a').text.strip())
            lineup_data.append({'Team': team_abbr,'Name': name, 'Position': pos })
        return(pd.DataFrame(lineup_data))
    
    def get_second_recent(element, class_name):
        previous_elements = element.find_all_previous('div', class_ = class_name)
        if len(previous_elements) > 1:
            return(previous_elements[1].text.strip())
        return(None)

    home_team_data = []
    for home_team_list in soup.select('.lineup__list.is-home'):
        team_abbr = home_team_list.find_previous('div', class_ = 'lineup__abbr').text.strip()
        home_team_data.append(extract_team_data(home_team_list, team_abbr))
    home_df = pd.concat(home_team_data)
    
    away_team_data = []
    for away_team_list in soup.select('.lineup__list.is-visit'):
        #team_abbr = away_team_list.find_previous('div', class_ = 'lineup__abbr').text.strip()
        team_abbr = get_second_recent(away_team_list, 'lineup__abbr')
        away_team_data.append(extract_team_data(away_team_list, team_abbr))
    away_df = pd.concat(away_team_data)
    
    final_df = pd.concat([home_df, away_df], ignore_index=True)
    return(final_df)

## Testing data acquisition before adding logic to main function

In [98]:
todays_lineups = get_daily_lineups1()

todays_lineups

,Team,Name,Position
0,CWS,Chase Meidroth,2B
1,CWS,Miguel Vargas,3B
2,CWS,Munetaka Murakami,1B
3,CWS,Austin Hays,DH
4,CWS,Colson Montgomery,SS
...,...,...,...
247,KC,Carter Jensen,C
248,KC,Michael Massey,2B
249,KC,Jac Caglianone,RF
250,KC,Isaac Collins,LF


In [13]:
batting_order = [1, 2, 3, 4, 5, 6, 7, 8, 9]

# Repeat that pattern enough times to cover every row in the table
todays_lineups['battingSpot'] = (batting_order * (len(todays_lineups) // 9 + 1))[:len(todays_lineups)]

In [14]:
todays_lineups

,Team,Name,Position,battingSpot
0,CIN,Dane Myers,CF,1
1,CIN,Matt McLain,2B,2
2,CIN,Elly De La Cruz,SS,3
3,CIN,Sal Stewart,1B,4
4,CIN,Eugenio Suarez,DH,5
...,...,...,...,...
247,MIA,Liam Hicks,C,5
248,MIA,Agustin Ramirez,DH,6
249,MIA,Owen Caissie,RF,7
250,MIA,Graham Pauley,3B,8


In [15]:
todays_lineups['Name'] = todays_lineups['Name'].str.lower()


In [16]:
abbr_conversion1 = {
    'WSH': 'WAS',
    'SD': 'SDP',
    'CWS': 'CHW',
    'SF': 'SFG',
    'TB': 'TBR',
    'KC': 'KCR'   
}
todays_lineups['order'] = range(1, len(todays_lineups) + 1)
todays_lineups['Team'] = todays_lineups['Team'].replace(abbr_conversion1)

In [17]:
todays_lineups = duckdb.query("""
    WITH name_counts AS (
        -- Step 1: Identify which names only appear once in player_dim
        SELECT player_name, COUNT(*) as name_occurrence
        FROM player_dim
        GROUP BY player_name
    )
    SELECT 
        tl.*,
        pd.fg_id, 
        pd.mlb_id
    FROM todays_lineups tl
    LEFT JOIN player_dim pd 
        ON tl.Name = pd.player_name
    LEFT JOIN name_counts nc
        ON tl.Name = nc.player_name
    WHERE 
        -- Rule 1: It's a match if the teams align
        tl.Team = pd.Team 
        OR 
        -- Rule 2: It's a match if that name only exists once in the whole dimension table
        nc.name_occurrence = 1
        OR
        -- Rule 3: Catch-all if we haven't assigned a team yet but need the record
        pd.Team IS NULL
    
    -- Step 2: Use QUALIFY to keep exactly ONE record per original lineup slot
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY tl.order 
        ORDER BY (tl.Team = pd.Team) DESC, pd.fg_id ASC
    ) = 1
    
    ORDER BY tl.order
""").df()

todays_lineups

,Team,Name,Position,battingSpot,order,fg_id,mlb_id
0,CIN,dane myers,CF,1,1,22054,667472
1,CIN,matt mclain,2B,2,2,29695,680574
2,CIN,elly de la cruz,SS,3,3,26668,682829
3,CIN,sal stewart,1B,4,4,31505,701398
4,CIN,eugenio suarez,DH,5,5,12552,553993
...,...,...,...,...,...,...,...
247,MIA,liam hicks,C,5,248,29844,689414
248,MIA,agustin ramirez,DH,6,249,26546,682663
249,MIA,owen caissie,RF,7,250,27496,683357
250,MIA,graham pauley,3B,8,251,31363,688363


In [84]:
# Function web scrapes daily pitching matchips from Rotowire

def get_daily_pitchers():
  url = 'https://www.rotowire.com/baseball/daily-lineups.php'
  response = requests.get(url)
  soup = BeautifulSoup(response.content, 'html.parser')
  
  def extract_team_data(team_list, team_abbr):
        pitcher_data = []
        for player in team_list.select('.lineup__player-highlight-name'):
            hand_elem = player.find('span', class_ = 'lineup__throws')
            hand = hand_elem.text.strip() if hand_elem else 'Unknown'
            name = player.find('a').get('href', player.find('a').text.strip())
            #name_elem = player.find('a')
            #name = name_elem.text.strip() if name_elem else 'Unknown'
            pitcher_data.append({'Team': team_abbr,'name_url': name, 'Hand': hand })
        return(pd.DataFrame(pitcher_data))
  
  def get_second_recent(element, class_name):
        previous_elements = element.find_all_previous('div', class_ = class_name)
        if len(previous_elements) > 1:
            return(previous_elements[1].text.strip())
        return(None)
  
  def clean_pitcher_names(urlstyle):
    player_part = urlstyle.split('/')[-1].split('-')[:-1]
    #name_parts = player_part.split('-')
    cleaned_name = ' '.join(part.capitalize() for part in player_part)
    return cleaned_name
    
  home_team_data = []
  for home_pitcher_list in soup.select('.lineup__list.is-home'):
    team_abbr = home_pitcher_list.find_previous('div', class_ = 'lineup__abbr').text.strip()
    home_team_data.append(extract_team_data(home_pitcher_list, team_abbr))
  if len(home_team_data) != 0:
    home_df = pd.concat(home_team_data)
  else:
    return('There are no games today.')
  
  away_team_data = []
  for away_pitcher_list in soup.select('.lineup__list.is-visit'):
     team_abbr = get_second_recent(away_pitcher_list, 'lineup__abbr')
     away_team_data.append(extract_team_data(away_pitcher_list, team_abbr))
  away_df = pd.concat(away_team_data)
  
  final_df = pd.concat([home_df, away_df], ignore_index= True)
  final_df['Name'] = final_df['name_url'].apply(clean_pitcher_names)
  final_df['Team'] = final_df['Team'].astype(str)
  return(final_df)


## More pre-implementation testing

In [99]:
todays_pitchers = get_daily_pitchers()

todays_pitchers

,Team,name_url,Hand,Name
0,CWS,/baseball/player/erick-fedde-13346,R,Erick Fedde
1,CLE,/baseball/player/gavin-williams-14876,R,Gavin Williams
2,MIN,/baseball/player/taj-bradley-15684,R,Taj Bradley
3,TEX,/baseball/player/nathan-eovaldi-10901,R,Nathan Eovaldi
4,TOR,/baseball/player/eric-lauer-14263,L,Eric Lauer
5,LAD,/baseball/player/tyler-glasnow-12248,R,Tyler Glasnow
6,SD,/baseball/player/matt-waldron-19015,R,Matt Waldron
7,PHI,/baseball/player/cristopher-sanchez-16500,L,Cristopher Sanchez
8,CIN,/baseball/player/brandon-williamson-16108,L,Brandon Williamson
9,PIT,/baseball/player/bubba-chandler-17314,R,Bubba Chandler


In [20]:
todays_pitchers['Team'] = todays_pitchers['Team'].replace(abbr_conversion1)
todays_pitchers['Name'] = todays_pitchers['Name'].str.lower()
todays_pitchers['order'] = range(1, len(todays_pitchers) + 1)

In [21]:
todays_pitchers = duckdb.query("""
    WITH name_counts AS (
        -- Step 1: Identify which names only appear once in player_dim
        SELECT player_name, COUNT(*) as name_occurrence
        FROM player_dim
        GROUP BY player_name
    )
    SELECT 
        tl.*,
        pd.fg_id, 
        pd.mlb_id
    FROM todays_pitchers tl
    LEFT JOIN player_dim pd 
        ON tl.Name = pd.player_name
    LEFT JOIN name_counts nc
        ON tl.Name = nc.player_name
    WHERE 
        -- Rule 1: It's a match if the teams align
        tl.Team = pd.Team 
        OR 
        -- Rule 2: It's a match if that name only exists once in the whole dimension table
        nc.name_occurrence = 1
        OR
        -- Rule 3: Catch-all if we haven't assigned a team yet but need the record
        pd.Team IS NULL
    
    -- Step 2: Use QUALIFY to keep exactly ONE record per original lineup slot
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY tl.order 
        ORDER BY (tl.Team = pd.Team) DESC, pd.fg_id ASC
    ) = 1

""").df()

todays_pitchers

,Team,name_url,Hand,Name,order,fg_id,mlb_id
0,CHW,/baseball/player/erick-fedde-13346,R,erick fedde,7,17425,607200
1,CHC,/baseball/player/jameson-taillon-11435,R,jameson taillon,27,11674,592791
2,TBR,/baseball/player/drew-rasmussen-14877,R,drew rasmussen,5,25385,656876
3,MIL,/baseball/player/brandon-woodruff-14525,R,brandon woodruff,9,16162,605540
4,SFG,/baseball/player/adrian-houser-12199,R,adrian houser,14,12718,605288
5,CLE,/baseball/player/gavin-williams-14876,R,gavin williams,17,30122,668909
6,HOU,/baseball/player/lance-mccullers-12459,R,lance mccullers,11,14120,621121
7,BOS,/baseball/player/brayan-bello-15839,R,brayan bello,16,23920,678394
8,SEA,/baseball/player/george-kirby-16057,R,george kirby,26,25436,669923
9,STL,/baseball/player/andre-pallante-16167,R,andre pallante,12,26108,669467


In [22]:
# Function web scrapes matchup information from Rotowire

def get_matchups():
    df = get_daily_pitchers()
    if type(df) == str:
        if df == 'There are no games today.':
            return('There are no games today.')
    else: 
        team_total = df.shape[0]
        team_total = int(team_total)
        game_count = df.shape[0] / 2
        game_count = int(game_count)
        df_edit = pd.DataFrame(df['Team'])
        home_teams = df_edit.iloc[:game_count]
        home_teams = home_teams.rename(columns = {'Team': 'HomeTeam'})
        away_teams = df_edit.iloc[game_count:team_total]
        away_teams = away_teams.rename(columns = {'Team': 'AwayTeam'})
        away_teams = away_teams.reset_index(drop=True)
        final_df = pd.concat([home_teams, away_teams], axis=1)
        final_df['AwayTeam'] = final_df['AwayTeam'].astype(str)
        final_df['HomeTeam'] = final_df['HomeTeam'].astype(str)
        #final_df = final_df[(final_df['HomeTeam']==team_abbr) | (final_df['AwayTeam']==team_abbr)]
        if final_df.empty:
            return('No games today')
        return(final_df)


## More implementation testing

In [100]:
todays_matchups = get_matchups()

todays_matchups

,HomeTeam,AwayTeam
0,CWS,LAA
1,CLE,TB
2,MIN,SEA
3,TEX,NYY
4,TOR,BOS
5,LAD,MIA
6,SD,CHC
7,PHI,SF
8,CIN,COL
9,PIT,STL


In [24]:
todays_matchups['HomeTeam'] = todays_matchups['HomeTeam'].replace(abbr_conversion1)
todays_matchups['AwayTeam'] = todays_matchups['AwayTeam'].replace(abbr_conversion1)

todays_matchups

,HomeTeam,AwayTeam
0,CIN,DET
1,BAL,BOS
2,TOR,CLE
3,NYM,COL
4,TBR,MIN
5,ATL,PHI
6,CHW,WAS
7,KCR,LAA
8,MIL,PIT
9,TEX,ATH


In [25]:
gbs_base1 = duckdb.query("""
    SELECT 
        m.HomeTeam,
        m.AwayTeam,
        -- Home Pitcher Data
        hp.Name as HomePitcherName,
        hp.Hand as HomePitcherHand,
        hp.fg_id as HomePitcherIDFG,
        hp.mlb_id as HomePitcherIDMLB,
        -- Away Pitcher Data
        ap.Name as AwayPitcherName,
        ap.Hand as AwayPitcherHand,
        ap.fg_id as AwayPitcherIDFG,
        ap.mlb_id as AwayPitcherIDMLB
    FROM todays_matchups m
    -- Join for the Home Pitcher
    LEFT JOIN todays_pitchers hp 
        ON m.HomeTeam = hp.Team
    -- Join for the Away Pitcher
    LEFT JOIN todays_pitchers ap 
        ON m.AwayTeam = ap.Team
""").df()
gbs_base1

,HomeTeam,AwayTeam,HomePitcherName,HomePitcherHand,HomePitcherIDFG,HomePitcherIDMLB,AwayPitcherName,AwayPitcherHand,AwayPitcherIDFG,AwayPitcherIDMLB
0,LAD,CHC,emmet sheehan,R,29839,686218,jameson taillon,R,11674,592791
1,TOR,CLE,max scherzer,R,3137,453286,gavin williams,R,30122,668909
2,BAL,BOS,brandon young,R,27819,687064,brayan bello,R,23920,678394
3,STL,SEA,andre pallante,R,26108,669467,george kirby,R,25436,669923
4,ATL,PHI,grant holmes,R,16944,656550,andrew painter,R,30091,691725
5,CHW,WAS,erick fedde,R,17425,607200,miles mikolas,R,9803,571945
6,HOU,NYY,lance mccullers,R,14120,621121,will warren,R,30182,701542
7,TBR,MIN,drew rasmussen,R,25385,656876,taj bradley,R,22543,671737
8,MIL,PIT,brandon woodruff,R,16162,605540,paul skenes,R,33677,694973
9,TEX,ATH,nathan eovaldi,R,9132,543135,luis severino,R,15890,622663


In [26]:
lineups_wide = duckdb.query("""
    WITH teams AS (
        SELECT DISTINCT Team FROM todays_lineups
    )
    SELECT 
        t.Team,
        -- Names
        b1.Name AS B1Name, b2.Name AS B2Name, b3.Name AS B3Name, 
        b4.Name AS B4Name, b5.Name AS B5Name, b6.Name AS B6Name, 
        b7.Name AS B7Name, b8.Name AS B8Name, b9.Name AS B9Name,
        
        -- FG IDs
        b1.fg_id AS B1IDFG, b2.fg_id AS B2IDFG, b3.fg_id AS B3IDFG, 
        b4.fg_id AS B4IDFG, b5.fg_id AS B5IDFG, b6.fg_id AS B6IDFG, 
        b7.fg_id AS B7IDFG, b8.fg_id AS B8IDFG, b9.fg_id AS B9IDFG,
        
        -- MLB IDs
        b1.mlb_id AS B1IDMLB, b2.mlb_id AS B2IDMLB, b3.mlb_id AS B3IDMLB, 
        b4.mlb_id AS B4IDMLB, b5.mlb_id AS B5IDMLB, b6.mlb_id AS B6IDMLB, 
        b7.mlb_id AS B7IDMLB, b8.mlb_id AS B8IDMLB, b9.mlb_id AS B9IDMLB,
        
        -- Positional IDs (Defensive)
        p1b.mlb_id AS IDMLB_1B, p2b.mlb_id AS IDMLB_2B, pss.mlb_id AS IDMLB_SS,
        p3b.mlb_id AS IDMLB_3B, pc.mlb_id  AS IDMLB_C,  plf.mlb_id AS IDMLB_LF,
        pcf.mlb_id AS IDMLB_CF, prf.mlb_id AS IDMLB_RF
        
    FROM teams t
    -- Batting Order Joins
    LEFT JOIN todays_lineups b1 ON t.Team = b1.Team AND CAST(b1.battingSpot AS INT) = 1
    LEFT JOIN todays_lineups b2 ON t.Team = b2.Team AND CAST(b2.battingSpot AS INT) = 2
    LEFT JOIN todays_lineups b3 ON t.Team = b3.Team AND CAST(b3.battingSpot AS INT) = 3
    LEFT JOIN todays_lineups b4 ON t.Team = b4.Team AND CAST(b4.battingSpot AS INT) = 4
    LEFT JOIN todays_lineups b5 ON t.Team = b5.Team AND CAST(b5.battingSpot AS INT) = 5
    LEFT JOIN todays_lineups b6 ON t.Team = b6.Team AND CAST(b6.battingSpot AS INT) = 6
    LEFT JOIN todays_lineups b7 ON t.Team = b7.Team AND CAST(b7.battingSpot AS INT) = 7
    LEFT JOIN todays_lineups b8 ON t.Team = b8.Team AND CAST(b8.battingSpot AS INT) = 8
    LEFT JOIN todays_lineups b9 ON t.Team = b9.Team AND CAST(b9.battingSpot AS INT) = 9
    
    -- Defensive Position Joins
    LEFT JOIN todays_lineups p1b ON t.Team = p1b.Team AND p1b.Position = '1B'
    LEFT JOIN todays_lineups p2b ON t.Team = p2b.Team AND p2b.Position = '2B'
    LEFT JOIN todays_lineups pss ON t.Team = pss.Team AND pss.Position = 'SS'
    LEFT JOIN todays_lineups p3b ON t.Team = p3b.Team AND p3b.Position = '3B'
    LEFT JOIN todays_lineups pc  ON t.Team = pc.Team  AND pc.Position = 'C'
    LEFT JOIN todays_lineups plf ON t.Team = plf.Team AND plf.Position = 'LF'
    LEFT JOIN todays_lineups pcf ON t.Team = pcf.Team AND pcf.Position = 'CF'
    LEFT JOIN todays_lineups prf ON t.Team = prf.Team AND prf.Position = 'RF'
""").df()

gbs_pre_stats = duckdb.query("""
    SELECT 
        g.*,
        -- Home Lineup (Aliases added for clarity)
        hl.B1Name AS HomeB1Name, hl.B2Name AS HomeB2Name, hl.B3Name AS HomeB3Name, 
        hl.B4Name AS HomeB4Name, hl.B5Name AS HomeB5Name, hl.B6Name AS HomeB6Name, 
        hl.B7Name AS HomeB7Name, hl.B8Name AS HomeB8Name, hl.B9Name AS HomeB9Name,
        hl.B1IDFG AS HomeBatter1IDFG, hl.B2IDFG AS HomeBatter2IDFG, hl.B3IDFG AS HomeBatter3IDFG, hl.B4IDFG AS HomeBatter4IDFG, hl.B5IDFG as HomeBatter5IDFG,
        hl.B6IDFG AS HomeBatter6IDFG, hl.B7IDFG AS HomeBatter7IDFG, hl.B8IDFG AS HomeBatter8IDFG, hl.B9IDFG AS HomeBatter9IDFG,
        hl.B1IDMLB AS HomeBatter1IDMLB, hl.B2IDMLB AS HomeBatter2IDMLB, hl.B3IDMLB AS HomeBatter3IDMLB, hl.B4IDMLB AS HomeBatter4IDMLB, hl.B5IDMLB AS HomeBatter5IDMLB,
        hl.B6IDMLB AS HomeBatter6IDMLB, hl.B7IDMLB AS HomeBatter7IDMLB, hl.B8IDMLB AS HomeBatter8IDMLB, hl.B9IDMLB AS HomeBatter9IDMLB,
        hl.IDMLB_1B AS Home1BIDMLB, hl.IDMLB_2B AS Home2BIDMLB, hl.IDMLB_SS AS HomeSSIDMLB,
        hl.IDMLB_3B AS Home3BIDMLB, hl.IDMLB_C AS HomeCIDMLB, hl.IDMLB_LF AS HomeLFIDMLB,
        hl.IDMLB_CF AS HomeCFIDMLB, hl.IDMLB_RF AS HomeRFIDMLB,
        
        -- Away Lineup
       al.B1Name AS AwayB1Name, al.B2Name AS AwayB2Name, al.B3Name AS AwayB3Name,
       al.B4Name AS AwayB4Name, al.B5Name AS AwayB5Name, al.B6Name AS AwayB6Name,
       al.B7Name AS AwayB7Name, al.B8Name AS AwayB8Name, al.B9Name AS AwayB9Name,
       al.B1IDFG AS AwayBatter1IDFG, al.B2IDFG AS AwayBatter2IDFG, al.B3IDFG AS AwayBatter3IDFG, al.B4IDFG AS AwayBatter4IDFG, al.B5IDFG as AwayBatter5IDFG,
       al.B6IDFG AS AwayBatter6IDFG, al.B7IDFG AS AwayBatter7IDFG, al.B8IDFG AS AwayBatter8IDFG, al.B9IDFG AS AwayBatter9IDFG,
       al.B1IDMLB AS AwayBatter1IDMLB, al.B2IDMLB AS AwayBatter2IDMLB, al.B3IDMLB AS AwayBatter3IDMLB, al.B4IDMLB AS AwayBatter4IDMLB, al.B5IDMLB AS AwayBatter5IDMLB,
       al.B6IDMLB AS AwayBatter6IDMLB, al.B7IDMLB AS AwayBatter7IDMLB, al.B8IDMLB AS AwayBatter8IDMLB, al.B9IDMLB AS AwayBatter9IDMLB,
       al.IDMLB_1B AS Away1BIDMLB, al.IDMLB_2B AS Away2BIDMLB, al.IDMLB_SS AS AwaySSIDMLB,
       al.IDMLB_3B AS Away3BIDMLB, al.IDMLB_C AS AwayCIDMLB, al.IDMLB_LF AS AwayLFIDMLB,
       al.IDMLB_CF AS AwayCFIDMLB, al.IDMLB_RF AS AwayRFIDMLB
    FROM gbs_base1 g
    LEFT JOIN lineups_wide hl ON g.HomeTeam = hl.Team
    LEFT JOIN lineups_wide al ON g.AwayTeam = al.Team
""").df()
gbs_pre_stats

,HomeTeam,AwayTeam,HomePitcherName,HomePitcherHand,HomePitcherIDFG,HomePitcherIDMLB,AwayPitcherName,AwayPitcherHand,AwayPitcherIDFG,AwayPitcherIDMLB,...,AwayBatter8IDMLB,AwayBatter9IDMLB,Away1BIDMLB,Away2BIDMLB,AwaySSIDMLB,Away3BIDMLB,AwayCIDMLB,AwayLFIDMLB,AwayCFIDMLB,AwayRFIDMLB
0,CIN,DET,andrew abbott,L,29911,671096,framber valdez,L,17295,664285,...,679529,595879,679529,650402,595879,805808,693307,682985,663837,672761
1,BAL,BOS,brandon young,R,27819,687064,brayan bello,R,23920,678394,...,702332,665966,575929,691785,596115,702332,665966,680776,678882,677800
2,TOR,CLE,max scherzer,R,3137,453286,gavin williams,R,30122,668909,...,666310,677587,700932,682877,677587,608070,666310,680757,682177,671655
3,NYM,COL,freddy peralta,R,18679,642547,michael lorenzen,R,14843,547179,...,650489,686668,681198,650489,678662,691720,696100,666160,686668,687859
4,TBR,MIN,drew rasmussen,R,25385,656876,taj bradley,R,22543,671737,...,668904,686797,665019,807712,686797,668904,680777,663616,621439,670242
5,ATL,PHI,grant holmes,R,16944,656550,andrew painter,R,30091,691725,...,702222,665561,547180,681082,607208,664761,665561,669016,702222,666969
6,CHW,WAS,erick fedde,R,17425,607200,miles mikolas,R,9803,571945,...,677588,660688,671277,683083,682928,691781,660688,695734,696285,695578
7,KCR,LAA,noah cameron,L,30184,702070,yusei kikuchi,L,20633,579328,...,681351,669326,694384,687093,687263,672724,681351,669326,545361,666176
8,MIL,PIT,brandon woodruff,R,16162,605540,paul skenes,R,33677,694973,...,804606,663698,687462,664040,804606,694377,663698,668804,665833,656811
9,TEX,ATH,nathan eovaldi,R,9132,543135,luis severino,R,15890,622663,...,671732,680869,701762,643446,805779,691777,669127,691016,680869,671732


# Player dim work

---------
---------
---------

In [10]:
player_dim = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/player_dim.csv')
player_dim

,player_name,retro_id,mlb_id,bref_id,fg_id,Team
0,Jose Fernandez,NaN,699912,NaN,28020,NaN
1,Justin Crawford,NaN,702222,NaN,31791,NaN
2,Foster Griffin,NaN,656492,NaN,16432,NaN
3,George Klassen,NaN,691946,NaN,33464,NaN
4,Nicky Lopez,lopen001,670032,lopezni01,19339,NaN
...,...,...,...,...,...,...
2517,Justin Hagenman,hagej002,663795,hagenju01,21546,NaN
2518,Victor Mederos,medev001,682989,medervi01,31533,NaN
2519,Gordon Graceffo,gracg001,700669,gracego01,29519,NaN
2520,Andrew Walters,walta001,689958,waltean01,33834,NaN


In [16]:
player_dim.loc[player_dim['player_name'] == 'C. J. Abrams', 'player_name'] = 'CJ Abrams'

# 2. Global Fix: Remove all periods from the player_name column
# .str.replace with regex=False is efficient for simple character removal
player_dim['player_name'] = player_dim['player_name'].str.replace('.', '', regex=False)

In [25]:
player_dim.loc[player_dim['player_name'] == 'Gerardo Perdomo', 'player_name'] = 'Geraldo Perdomo'

In [28]:
player_dim.to_csv('player_dim.csv', index=False)

In [29]:
import re

# This regex finds a single letter [A-Z] followed by a space
# and replaces it with just the letter, effectively "closing the gap"
# Example: 'J P France' -> 'JP France'
# Example: 'A J Puk' -> 'AJ Puk'
pattern = r'([A-Za-z])\s(?=[A-Za-z]\b|(?=[A-Za-z]\s))'

# Apply to player_dim
player_dim['player_name'] = player_dim['player_name'].str.replace(pattern, r'\1', regex=True)

In [30]:
player_dim['player_name'] = player_dim['player_name'].str.lower()

In [38]:
# Filter for records where 'player_name' does NOT contain a space
no_spaces = player_dim[~player_dim['player_name'].str.contains(' ', na=False)]

# Display the results
print(f"Found {len(no_spaces)} records with no spaces:")
print(no_spaces[['player_name', 'mlb_id', 'fg_id']])

# Optional: If you want to see if these are just 'nan' strings or actual names
# like 'ichiro', this will help you spot them.

Found 0 records with no spaces:
Empty DataFrame
Columns: [player_name, mlb_id, fg_id]
Index: []


In [37]:
apostrophe_fixes = {
    657247: "brian o'keefe",
    676617: "riley o'brien",
    681351: "logan o'hoppe",
    657434: "brian o'grady",
    656811: "ryan o'hearn",
    641933: "tyler o'neill",
    598284: "peter o'brien",
    518595: "travis d'arnaud",
    488818: "chase d'arnaud"
}

# Apply the fixes to player_dim
for mlbid, correct_name in apostrophe_fixes.items():
    player_dim.loc[player_dim['mlb_id'] == mlbid, 'player_name'] = correct_name

In [51]:
player_dim.loc[player_dim['fg_id'] == '20391', 'Team'] = 'WAS'

In [106]:
player_dim.to_csv('player_dim_live.csv', index=False)

In [10]:
player_dim = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/player_dim_live.csv')

In [128]:
player_dim[player_dim['fg_id'] == '15491']

,player_name,retro_id,mlb_id,bref_id,fg_id,Team
341,jp crawford,crawj002,641487,crawfjp01,15491,NaN


In [18]:
player_dim.loc[player_dim['player_name'] == 'jp crawford', 'player_name'] = 'j.p. crawford'

# Stat Loading Sample: This is largely reloading statistic tables and restating join logic.

In [27]:
master_wins = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/master_wins.csv')
master_wins

,GAMEID,Date,AwayTeam,HomeTeam,AwayScore,HomeScore,HomeTeamWin,AwayTeamWin,Year,DeltaTeamWinsL162,...,DeltaB3_BSR_L2,DeltaB4_BSR_L2,DeltaB5_BSR_L2,DeltaB6_BSR_L2,DeltaB7_BSR_L2,DeltaB8_BSR_L2,DeltaB9_BSR_L2,DeltaBP_SO9_L3M,DeltaBP_BB9_L3M,DeltaBP_ERA_L3M
0,1.0,2018-03-29,NYY,TOR,6,1,0,1,2018.0,-15.0,...,0.2434,5.6442,-12.8442,-2.1093,-2.6026,9.4756,-0.8805,-2.184378,-0.534860,1.592203
1,2.0,2018-03-29,HOU,TEX,4,1,0,1,2018.0,-23.0,...,0.9189,-9.1525,-6.1101,-2.7385,-2.1698,1.4905,7.4531,-2.323342,0.495427,0.501419
2,3.0,2018-03-29,STL,NYM,4,9,1,0,2018.0,-13.0,...,-1.6255,-3.9495,-2.8930,-1.7338,2.1942,-4.7355,0.3276,-0.302129,1.393275,1.429524
3,4.0,2018-03-29,COL,ARI,2,8,1,0,2018.0,6.0,...,7.8618,-0.3334,5.8953,-7.4631,4.6734,15.9490,-0.7794,-0.091132,0.153348,-0.035633
4,5.0,2018-03-29,PHI,ATL,5,8,1,0,2018.0,6.0,...,1.3811,-3.0351,-4.0434,0.8642,5.6223,-0.2995,0.7911,-0.675361,0.833576,1.407952
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18278,NaN,2026-04-22,PHI,CHC,2,7,1,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18279,NaN,2026-04-22,PIT,TEX,8,4,0,1,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18280,NaN,2026-04-22,SDP,COL,3,8,1,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
18281,NaN,2026-04-22,CHW,ARI,7,11,1,0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
# Pivot master_wins to a long "Team-Centric" view
team_history = duckdb.query("""
    SELECT Date, AwayTeam as Team, AwayScore as Scored, HomeScore as Allowed, AwayTeamWin as Win FROM master_wins
    UNION ALL
    SELECT Date, HomeTeam as Team, HomeScore as Scored, AwayScore as Allowed, HomeTeamWin as Win FROM master_wins
""").df()

latest_rolling = duckdb.query("""
    SELECT 
        Team,
        SUM(Win) OVER(PARTITION BY Team ORDER BY Date ROWS BETWEEN 162 PRECEDING AND 1 PRECEDING) as WinsL162,
        SUM(Win) OVER(PARTITION BY Team ORDER BY Date ROWS BETWEEN 50 PRECEDING AND 1 PRECEDING) as WinsL50,
        SUM(Scored) OVER(PARTITION BY Team ORDER BY Date ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING) as RunsScoredL100,
        SUM(Allowed) OVER(PARTITION BY Team ORDER BY Date ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING) as RunsAllowedL100
    FROM team_history
    QUALIFY row_number() OVER(PARTITION BY Team ORDER BY Date DESC) = 1
""").df()

gbs_with_wins = duckdb.query("""
    SELECT 
        g.*,
        -- Away Team Features
        al.WinsL162 AS AwayTeamWinsL162,
        al.WinsL50 AS AwayTeamWinsL50,
        al.RunsScoredL100 AS AwayTeamRunsScoredL100,
        al.RunsAllowedL100 AS AwayTeamRunsAllowedL100,
        
        -- Home Team Features
        hl.WinsL162 AS HomeTeamWinsL162,
        hl.WinsL50 AS HomeTeamWinsL50,
        hl.RunsScoredL100 AS HomeTeamRunsScoredL100,
        hl.RunsAllowedL100 AS HomeTeamRunsAllowedL100
    FROM gbs_pre_stats g
    LEFT JOIN latest_rolling al ON g.AwayTeam = al.Team
    LEFT JOIN latest_rolling hl ON g.HomeTeam = hl.Team
""").df()

print(f"Features Engineered. Column Count: {len(gbs_with_wins.columns)}")
print(gbs_with_wins[['HomeTeam', 'HomeTeamWinsL50', 'AwayTeam', 'AwayTeamWinsL50']].head())

Features Engineered. Column Count: 88
  HomeTeam  HomeTeamWinsL50 AwayTeam  AwayTeamWinsL50
0      CIN             31.0      DET             20.0
1      BAL             25.0      BOS             23.0
2      TOR             25.0      CLE             35.0
3      NYM             17.0      COL             14.0
4      TBR             23.0      MIN             21.0


In [29]:
from datetime import datetime

today_val = pd.to_datetime('today').date()
year_val = int(today_val.year)

gbs_with_wins.insert(0, 'Date', today_val)
gbs_with_wins.insert(1, 'Year', year_val)

gbs_with_wins['Date'] = pd.to_datetime(gbs_with_wins['Date'])

### Pitching and Hitting

In [ ]:
SPGameLogFact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/SPGameLogFactCombined.csv')
BatterGameLogFact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/Batter_Combined_Fact.csv')
dra_map_clean = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/dra_map_clean.csv')
drc_map_clean = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/drc_map_clean.csv')
drcp_map_clean = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/drcp_map_clean.csv')
team_bullpen_monthly = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/team_bullpen_monthly.csv')
team_bullpen_monthly['CompDate'] = (
    pd.to_datetime(team_bullpen_monthly['query_year'].astype(str) + '-' + team_bullpen_monthly['query_month'].astype(str) + '-01')
    + pd.offsets.MonthEnd(0)
)
fangraphs_hitting_fact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/fg_hitting_fact.csv')
fangraphs_pitching_fact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/fangraphs_pitching_fact.csv')
defense_fact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/defense_fact.csv')

In [30]:
SPGameLogFact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/SPGameLogFactCombined.csv')

In [31]:
GameBoxScoresSPG = duckdb.query("""
with stats_weighted as (
    SELECT
        playerid,
        season,
        IP,
        Events,
        Date,
        ("K/BB" * IP) as K_BB_W,
        (xERA * IP) as xERA_W,
        ("Barrel%" * Events) as Barrel_W
    FROM SPGameLogFact
)

select GBS.*,
    hp.HomeSP_IP_L4A,
    hp.HomeSP_K_BB_L4A,
    hp.HomeSP_xERA_L4A,
    hp.HomeSP_Barrel_pct_L4A,

    ap.AwaySP_IP_L4A,
    ap.AwaySP_K_BB_L4A,
    ap.AwaySP_xERA_L4A,
    ap.AwaySP_Barrel_pct_L4A

FROM gbs_with_wins GBS

LEFT JOIN LATERAL(
    select 
    SUM(IP) as HomeSP_IP_L4A,
    SUM(K_BB_W) / NULLIF(SUM(IP), 0) as HomeSP_K_BB_L4A,
    SUM(xERA_W) / NULLIF(SUM(IP), 0) as HomeSP_xERA_L4A,
    SUM(Barrel_W) / NULLIF(SUM(Events), 0) as HomeSP_Barrel_pct_L4A
    FROM(
        SELECT
        IP,
        Events,
        K_BB_W,
        xERA_W,
        Barrel_W
        from stats_weighted sw
        where sw.playerid = GBS.HomePitcherIDFG
        and sw.season = GBS.Year
        and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC
        limit 4
    )
) hp on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(IP) as AwaySP_IP_L4A,
    SUM(K_BB_W) / NULLIF(SUM(IP), 0) as AwaySP_K_BB_L4A,
    SUM(xERA_W) / NULLIF(SUM(IP), 0) as AwaySP_xERA_L4A,
    SUM(Barrel_W) / NULLIF(SUM(Events), 0) as AwaySP_Barrel_pct_L4A
    FROM(
        SELECT
        IP,
        Events,
        K_BB_W,
        xERA_W,
        Barrel_W,
        from stats_weighted sw
        where sw.playerid = GBS.AwayPitcherIDFG
        and sw.season = GBS.Year
        and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC
        limit 4
    )
) ap on TRUE

""").df()

In [32]:
BatterGameLogFact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/Batter_Combined_Fact.csv')

In [33]:
GameBoxScoresSPG = duckdb.query("""
with stats_weighted as (
    SELECT
        playerid, season, AB, PA, Events, Date, 
        ("BB/K" * PA) as BB_K_W, 
        ("wRC+" * PA) as wRC_plus_W, 
        (xAVG * AB) as xAVG_W, 
        (xSLG * AB) as xSLG_W
    FROM BatterGameLogFact
)

select GBS.*,
    HomeB1_PA_L10A, HomeB1_BB_K_L10A, HomeB1_wRC_plus_L10A, HomeB1_xAVG_L10A, HomeB1_xSLG_L10A,
    HomeB2_PA_L10A, HomeB2_BB_K_L10A, HomeB2_wRC_plus_L10A, HomeB2_xAVG_L10A, HomeB2_xSLG_L10A,
    HomeB3_PA_L10A, HomeB3_BB_K_L10A, HomeB3_wRC_plus_L10A, HomeB3_xAVG_L10A, HomeB3_xSLG_L10A,
    HomeB4_PA_L10A, HomeB4_BB_K_L10A, HomeB4_wRC_plus_L10A, HomeB4_xAVG_L10A, HomeB4_xSLG_L10A,
    HomeB5_PA_L10A, HomeB5_BB_K_L10A, HomeB5_wRC_plus_L10A, HomeB5_xAVG_L10A, HomeB5_xSLG_L10A,
    HomeB6_PA_L10A, HomeB6_BB_K_L10A, HomeB6_wRC_plus_L10A, HomeB6_xAVG_L10A, HomeB6_xSLG_L10A,
    HomeB7_PA_L10A, HomeB7_BB_K_L10A, HomeB7_wRC_plus_L10A, HomeB7_xAVG_L10A, HomeB7_xSLG_L10A,
    HomeB8_PA_L10A, HomeB8_BB_K_L10A, HomeB8_wRC_plus_L10A, HomeB8_xAVG_L10A, HomeB8_xSLG_L10A,
    HomeB9_PA_L10A, HomeB9_BB_K_L10A, HomeB9_wRC_plus_L10A, HomeB9_xAVG_L10A, HomeB9_xSLG_L10A,
    
    AwayB1_PA_L10A, AwayB1_BB_K_L10A, AwayB1_wRC_plus_L10A, AwayB1_xAVG_L10A, AwayB1_xSLG_L10A,
    AwayB2_PA_L10A, AwayB2_BB_K_L10A, AwayB2_wRC_plus_L10A, AwayB2_xAVG_L10A, AwayB2_xSLG_L10A,
    AwayB3_PA_L10A, AwayB3_BB_K_L10A, AwayB3_wRC_plus_L10A, AwayB3_xAVG_L10A, AwayB3_xSLG_L10A,
    AwayB4_PA_L10A, AwayB4_BB_K_L10A, AwayB4_wRC_plus_L10A, AwayB4_xAVG_L10A, AwayB4_xSLG_L10A,
    AwayB5_PA_L10A, AwayB5_BB_K_L10A, AwayB5_wRC_plus_L10A, AwayB5_xAVG_L10A, AwayB5_xSLG_L10A,
    AwayB6_PA_L10A, AwayB6_BB_K_L10A, AwayB6_wRC_plus_L10A, AwayB6_xAVG_L10A, AwayB6_xSLG_L10A,
    AwayB7_PA_L10A, AwayB7_BB_K_L10A, AwayB7_wRC_plus_L10A, AwayB7_xAVG_L10A, AwayB7_xSLG_L10A,
    AwayB8_PA_L10A, AwayB8_BB_K_L10A, AwayB8_wRC_plus_L10A, AwayB8_xAVG_L10A, AwayB8_xSLG_L10A,
    AwayB9_PA_L10A, AwayB9_BB_K_L10A, AwayB9_wRC_plus_L10A, AwayB9_xAVG_L10A, AwayB9_xSLG_L10A

FROM GameBoxScoresSPG GBS

LEFT JOIN LATERAL(
    select 
        SUM(PA) as HomeB1_PA_L10A,
        SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB1_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB1_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB1_xAVG_L10A,
        SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB1_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.HomeBatter1IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) hb1 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as HomeB2_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB2_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB2_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB2_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB2_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.HomeBatter2IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) hb2 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as HomeB3_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB3_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB3_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB3_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB3_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.HomeBatter3IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) hb3 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as HomeB4_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB4_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB4_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB4_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB4_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.HomeBatter4IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) hb4 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as HomeB5_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB5_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB5_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB5_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB5_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.HomeBatter5IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) hb5 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as HomeB6_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB6_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB6_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB6_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB6_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.HomeBatter6IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) hb6 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as HomeB7_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB7_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB7_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB7_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB7_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.HomeBatter7IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) hb7 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as HomeB8_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB8_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB8_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB8_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB8_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.HomeBatter8IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) hb8 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as HomeB9_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB9_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB9_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB9_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB9_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.HomeBatter9IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) hb9 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as AwayB1_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB1_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB1_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB1_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB1_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.AwayBatter1IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) ab1 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as AwayB2_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB2_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB2_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB2_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB2_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.AwayBatter2IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) ab2 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as AwayB3_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB3_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB3_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB3_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB3_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.AwayBatter3IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) ab3 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as AwayB4_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB4_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB4_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB4_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB4_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.AwayBatter4IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) ab4 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as AwayB5_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB5_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB5_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB5_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB5_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.AwayBatter5IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) ab5 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as AwayB6_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB6_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB6_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB6_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB6_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.AwayBatter6IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) ab6 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as AwayB7_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB7_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB7_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB7_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB7_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.AwayBatter7IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) ab7 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as AwayB8_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB8_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB8_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB8_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB8_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.AwayBatter8IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) ab8 on TRUE

LEFT JOIN LATERAL(
    select 
        SUM(PA) as AwayB9_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB9_BB_K_L10A,
        SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB9_wRC_plus_L10A,
        SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB9_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB9_xSLG_L10A
    FROM(
        SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
        from stats_weighted sw
        where sw.playerid = GBS.AwayBatter9IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
        order by sw.Date DESC limit 10
    )
) ab9 on TRUE

""").df()

In [34]:
dra_map_clean = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/dra_map_clean.csv')

In [35]:
GameBoxScoresSPG = duckdb.query("""
with stats_weighted as (
        SELECT
        bpid,
        mlbid,
        Year,
        Month,
        IP,
        CompDate,
        (DRA * IP) as DRA_W,
        ("K%" * IP) as Kpct_W,
        ("BB%" * IP) as BBpct_W
    FROM dra_map_clean
)


select GBS.*,
hp.HomeSP_IP_L5M,
hp.HomeSP_DRA_L5M,
hp.HomeSP_Kpct_L5M,
hp.HomeSP_BBpct_L5M,
ap.AwaySP_IP_L5M,
ap.AwaySP_DRA_L5M,
ap.AwaySP_Kpct_L5M,
ap.AwaySP_BBpct_L5M

FROM GameBoxScoresSPG GBS

LEFT JOIN LATERAL(
    select 
    SUM(IP) as HomeSP_IP_L5M,
    SUM(DRA_W) / NULLIF(SUM(IP), 0) as HomeSP_DRA_L5M,
    SUM(Kpct_W) / NULLIF(SUM(IP), 0) as HomeSP_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(IP), 0) as HomeSP_BBpct_L5M
    FROM(
        SELECT
        IP,
        DRA_W,
        Kpct_W,
        BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomePitcherIDMLB
        and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) hp on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(IP) as AwaySP_IP_L5M,
    SUM(DRA_W) / NULLIF(SUM(IP), 0) as AwaySP_DRA_L5M,
    SUM(Kpct_W) / NULLIF(SUM(IP), 0) as AwaySP_Kpct_L5M,
    SUM(BBpct_W) / NULLIF(SUM(IP), 0) as AwaySP_BBpct_L5M
    FROM(
        SELECT
        IP,
        DRA_W,
        Kpct_W,
        BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayPitcherIDMLB
        and CAST(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 5
    )
) ap on TRUE

""").df()

In [36]:
drc_map_clean = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/drc_map_clean.csv')
drcp_map_clean = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/drcp_map_clean.csv')

In [37]:
GameBoxScores_bphitting = duckdb.query("""
with stats_weighted as (
        SELECT bpid, mlbid, Year, Month, PA, CompDate, (OPS * PA) as OPS_W, (DRC_plus * PA) as DRC_plus_W,
        ("K%" * PA) as Kpct_W
    FROM drc_map_clean
)

select GBS.*,

hb1.HomeB1_PA_L5M,hb1.HomeB1_OPS_L5M,hb1.HomeB1_DRC_plus_L5M,hb1.HomeB1_Kpct_L5M,
hb2.HomeB2_PA_L5M,hb2.HomeB2_OPS_L5M,hb2.HomeB2_DRC_plus_L5M,hb2.HomeB2_Kpct_L5M,
hb3.HomeB3_PA_L5M,hb3.HomeB3_OPS_L5M,hb3.HomeB3_DRC_plus_L5M,hb3.HomeB3_Kpct_L5M,
hb4.HomeB4_PA_L5M,hb4.HomeB4_OPS_L5M,hb4.HomeB4_DRC_plus_L5M,hb4.HomeB4_Kpct_L5M,
hb5.HomeB5_PA_L5M,hb5.HomeB5_OPS_L5M,hb5.HomeB5_DRC_plus_L5M,hb5.HomeB5_Kpct_L5M,
hb6.HomeB6_PA_L5M,hb6.HomeB6_OPS_L5M,hb6.HomeB6_DRC_plus_L5M,hb6.HomeB6_Kpct_L5M,
hb7.HomeB7_PA_L5M,hb7.HomeB7_OPS_L5M,hb7.HomeB7_DRC_plus_L5M,hb7.HomeB7_Kpct_L5M,
hb8.HomeB8_PA_L5M,hb8.HomeB8_OPS_L5M,hb8.HomeB8_DRC_plus_L5M,hb8.HomeB8_Kpct_L5M,
hb9.HomeB9_PA_L5M,hb9.HomeB9_OPS_L5M,hb9.HomeB9_DRC_plus_L5M,hb9.HomeB9_Kpct_L5M,

ab1.AwayB1_PA_L5M,ab1.AwayB1_OPS_L5M,ab1.AwayB1_DRC_plus_L5M,ab1.AwayB1_Kpct_L5M,
ab2.AwayB2_PA_L5M,ab2.AwayB2_OPS_L5M,ab2.AwayB2_DRC_plus_L5M,ab2.AwayB2_Kpct_L5M,
ab3.AwayB3_PA_L5M,ab3.AwayB3_OPS_L5M,ab3.AwayB3_DRC_plus_L5M,ab3.AwayB3_Kpct_L5M,
ab4.AwayB4_PA_L5M,ab4.AwayB4_OPS_L5M,ab4.AwayB4_DRC_plus_L5M,ab4.AwayB4_Kpct_L5M,
ab5.AwayB5_PA_L5M,ab5.AwayB5_OPS_L5M,ab5.AwayB5_DRC_plus_L5M,ab5.AwayB5_Kpct_L5M,
ab6.AwayB6_PA_L5M,ab6.AwayB6_OPS_L5M,ab6.AwayB6_DRC_plus_L5M,ab6.AwayB6_Kpct_L5M,
ab7.AwayB7_PA_L5M,ab7.AwayB7_OPS_L5M,ab7.AwayB7_DRC_plus_L5M,ab7.AwayB7_Kpct_L5M,
ab8.AwayB8_PA_L5M,ab8.AwayB8_OPS_L5M,ab8.AwayB8_DRC_plus_L5M,ab8.AwayB8_Kpct_L5M,
ab9.AwayB9_PA_L5M,ab9.AwayB9_OPS_L5M,ab9.AwayB9_DRC_plus_L5M,ab9.AwayB9_Kpct_L5M

FROM GameBoxScoresSPG GBS

LEFT JOIN LATERAL (
    select 
    SUM(PA) as HomeB1_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB1_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB1_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB1_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter1IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) hb1 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as HomeB2_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB2_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB2_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB2_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter2IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) hb2 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as HomeB3_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB3_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB3_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB3_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter3IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) hb3 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as HomeB4_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB4_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB4_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB4_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter4IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) hb4 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as HomeB5_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB5_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB5_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB5_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter5IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) hb5 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as HomeB6_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB6_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB6_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB6_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter6IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) hb6 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as HomeB7_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB7_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB7_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB7_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter7IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) hb7 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as HomeB8_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB8_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB8_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB8_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter8IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) hb8 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as HomeB9_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB9_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB9_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB9_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter9IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) hb9 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as AwayB1_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB1_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB1_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB1_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter1IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) ab1 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as AwayB2_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB2_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB2_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB2_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter2IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) ab2 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as AwayB3_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB3_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB3_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB3_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter3IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) ab3 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as AwayB4_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB4_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB4_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB4_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter4IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) ab4 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as AwayB5_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB5_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB5_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB5_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter5IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) ab5 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as AwayB6_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB6_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB6_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB6_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter6IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) ab6 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as AwayB7_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB7_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB7_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB7_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter7IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) ab7 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as AwayB8_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB8_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB8_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB8_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter8IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) ab8 on TRUE

LEFT JOIN LATERAL (
    select 
    SUM(PA) as AwayB9_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB9_OPS_L5M,
    SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB9_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB9_Kpct_L5M
    FROM (
        SELECT PA,OPS_W,DRC_plus_W,Kpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter9IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC limit 5
    )
) ab9 on TRUE

""").df()

In [38]:
GameBoxScores_phitting = duckdb.query("""
with stats_weighted as (
        SELECT bpid, mlbid, Year, Month, PlatoonHand, PA, CompDate, (OPS * PA) as OPS_W,
        ("BB%" * PA) as BBpct_W
    FROM drcp_map_clean
)

select GBS.*,

hb1.HomeB1_PA_L5M_P,hb1.HomeB1_OPS_L5M_P,hb1.HomeB1_BBpct_L5M_P,
hb2.HomeB2_PA_L5M_P,hb2.HomeB2_OPS_L5M_P,hb2.HomeB2_BBpct_L5M_P,
hb3.HomeB3_PA_L5M_P,hb3.HomeB3_OPS_L5M_P,hb3.HomeB3_BBpct_L5M_P,
hb4.HomeB4_PA_L5M_P,hb4.HomeB4_OPS_L5M_P,hb4.HomeB4_BBpct_L5M_P,
hb5.HomeB5_PA_L5M_P,hb5.HomeB5_OPS_L5M_P,hb5.HomeB5_BBpct_L5M_P,
hb6.HomeB6_PA_L5M_P,hb6.HomeB6_OPS_L5M_P,hb6.HomeB6_BBpct_L5M_P,
hb7.HomeB7_PA_L5M_P,hb7.HomeB7_OPS_L5M_P,hb7.HomeB7_BBpct_L5M_P,
hb8.HomeB8_PA_L5M_P,hb8.HomeB8_OPS_L5M_P,hb8.HomeB8_BBpct_L5M_P,
hb9.HomeB9_PA_L5M_P,hb9.HomeB9_OPS_L5M_P,hb9.HomeB9_BBpct_L5M_P,

ab1.AwayB1_PA_L5M_P,ab1.AwayB1_OPS_L5M_P,ab1.AwayB1_BBpct_L5M_P,
ab2.AwayB2_PA_L5M_P,ab2.AwayB2_OPS_L5M_P,ab2.AwayB2_BBpct_L5M_P,
ab3.AwayB3_PA_L5M_P,ab3.AwayB3_OPS_L5M_P,ab3.AwayB3_BBpct_L5M_P,
ab4.AwayB4_PA_L5M_P,ab4.AwayB4_OPS_L5M_P,ab4.AwayB4_BBpct_L5M_P,
ab5.AwayB5_PA_L5M_P,ab5.AwayB5_OPS_L5M_P,ab5.AwayB5_BBpct_L5M_P,
ab6.AwayB6_PA_L5M_P,ab6.AwayB6_OPS_L5M_P,ab6.AwayB6_BBpct_L5M_P,
ab7.AwayB7_PA_L5M_P,ab7.AwayB7_OPS_L5M_P,ab7.AwayB7_BBpct_L5M_P,
ab8.AwayB8_PA_L5M_P,ab8.AwayB8_OPS_L5M_P,ab8.AwayB8_BBpct_L5M_P,
ab9.AwayB9_PA_L5M_P,ab9.AwayB9_OPS_L5M_P,ab9.AwayB9_BBpct_L5M_P

FROM GameBoxScores_bphitting GBS

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB1_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB1_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB1_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter1IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb1 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB2_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB2_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB2_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter2IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb2 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB3_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB3_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB3_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter3IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb3 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB4_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB4_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB4_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter4IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb4 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB5_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB5_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB5_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter5IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb5 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB6_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB6_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB6_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter6IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb6 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB7_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB7_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB7_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter7IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb7 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB8_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB8_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB8_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter8IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb8 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as HomeB9_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB9_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB9_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.HomeBatter9IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) hb9 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB1_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB1_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB1_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter1IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab1 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB2_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB2_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB2_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter2IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab2 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB3_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB3_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB3_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter3IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab3 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB4_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB4_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB4_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter4IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab4 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB5_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB5_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB5_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter5IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab5 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB6_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB6_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB6_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter6IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab6 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB7_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB7_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB7_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter7IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab7 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB8_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB8_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB8_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter8IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab8 on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(PA) as AwayB9_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB9_OPS_L5M_P,
    SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB9_BBpct_L5M_P
    FROM(
        SELECT
        PA,OPS_W,BBpct_W
        from stats_weighted sw
        where sw.mlbid = GBS.AwayBatter9IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
        order by sw.CompDate DESC
        limit 5
    )
) ab9 on TRUE

""").df()

In [39]:
team_bullpen_monthly = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/team_bullpen_monthly.csv')
team_bullpen_monthly['CompDate'] = (
    pd.to_datetime(team_bullpen_monthly['query_year'].astype(str) + '-' + team_bullpen_monthly['query_month'].astype(str) + '-01')
    + pd.offsets.MonthEnd(0)
)

In [40]:
GameBoxScores_w_bullpen = duckdb.query("""
with stats_weighted as (
        SELECT
        Team_Abbr,
        query_year,
        query_month,
        IP,
        CompDate,
        (ERA * IP) as ERA_W,
        (SO9 * IP) as SO9_W,
        (BB9 * IP) as BB9_W
    FROM team_bullpen_monthly
)


select GBS.*,
hp.HomeBP_ERA_L3M,
hp.HomeBP_SO9_L3M,
hp.HomeBP_BB9_L3M,
ap.AwayBP_ERA_L3M,
ap.AwayBP_SO9_L3M,
ap.AwayBP_BB9_L3M

FROM GameBoxScores_phitting GBS

LEFT JOIN LATERAL(
    select 
    SUM(ERA_W) / NULLIF(SUM(IP), 0) as HomeBP_ERA_L3M,
    SUM(SO9_W) / NULLIF(SUM(IP), 0) as HomeBP_SO9_L3M,
    SUM(BB9_W) / NULLIF(SUM(IP), 0) as HomeBP_BB9_L3M
    FROM(
        SELECT
        IP,
        ERA_W,
        SO9_W,
        BB9_W
        from stats_weighted sw
        where sw.Team_Abbr = GBS.HomeTeam
        and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 3
    )
) hp on TRUE

LEFT JOIN LATERAL(
    select 
    SUM(ERA_W) / NULLIF(SUM(IP), 0) as AwayBP_ERA_L3M,
    SUM(SO9_W) / NULLIF(SUM(IP), 0) as AwayBP_SO9_L3M,
    SUM(BB9_W) / NULLIF(SUM(IP), 0) as AwayBP_BB9_L3M
    FROM(
        SELECT
        IP,
        ERA_W,
        SO9_W,
        BB9_W
        from stats_weighted sw
        where sw.Team_Abbr = GBS.AwayTeam
        and cast(sw.CompDate as date) < cast(GBS.Date as date)
        order by sw.CompDate DESC
        limit 3
    )
) ap on TRUE

""").df()

In [41]:
GameBoxScores_w_bullpen

,Date,Year,HomeTeam,AwayTeam,HomePitcherName,HomePitcherHand,HomePitcherIDFG,HomePitcherIDMLB,AwayPitcherName,AwayPitcherHand,...,AwayB8_BBpct_L5M_P,AwayB9_PA_L5M_P,AwayB9_OPS_L5M_P,AwayB9_BBpct_L5M_P,HomeBP_ERA_L3M,HomeBP_SO9_L3M,HomeBP_BB9_L3M,AwayBP_ERA_L3M,AwayBP_SO9_L3M,AwayBP_BB9_L3M
0,2026-04-24,2026,CIN,DET,andrew abbott,L,29911,671096,framber valdez,L,...,14.371233,102.0,0.777118,0.988235,3.737094,9.729912,4.107437,4.216507,7.924641,3.857656
1,2026-04-24,2026,LAD,CHC,emmet sheehan,R,29839,686218,jameson taillon,R,...,4.986981,395.0,0.719258,8.102532,4.493918,9.832557,4.324974,4.289832,9.810080,2.593852
2,2026-04-24,2026,TBR,MIN,drew rasmussen,R,25385,656876,taj bradley,R,...,5.627465,310.0,0.619045,6.135484,3.916116,11.041680,3.121115,4.980583,7.805825,3.378641
3,2026-04-24,2026,ATL,PHI,grant holmes,R,16944,656550,andrew painter,R,...,NaN,71.0,0.577620,4.233803,4.671119,8.532577,3.176361,4.107586,9.252038,3.070720
4,2026-04-24,2026,CHW,WAS,erick fedde,R,17425,607200,miles mikolas,R,...,7.452632,275.0,0.601251,3.274182,4.292482,9.484064,4.118462,5.341333,8.727096,4.115453
5,2026-04-24,2026,NYM,COL,freddy peralta,R,18679,642547,michael lorenzen,R,...,10.902465,336.0,0.590167,5.358631,4.585919,8.416509,3.237119,5.859643,8.126943,3.798462
6,2026-04-24,2026,STL,SEA,andre pallante,R,26108,669467,george kirby,R,...,11.043684,115.0,0.699348,18.269565,3.825784,9.219512,3.418118,3.927835,9.247423,2.814433
7,2026-04-24,2026,KCR,LAA,noah cameron,L,30184,702070,yusei kikuchi,L,...,7.369118,28.0,0.772036,3.589286,3.516003,7.709401,3.161177,4.599708,9.085137,3.799759
8,2026-04-24,2026,HOU,NYY,lance mccullers,R,14120,621121,will warren,R,...,11.691707,377.0,0.781416,12.209814,3.878162,9.384560,3.611723,5.138369,9.622168,3.731045
9,2026-04-24,2026,BAL,BOS,brandon young,R,27819,687064,brayan bello,R,...,6.235503,253.0,0.741755,7.884190,4.561489,8.758059,4.087094,3.257463,8.563433,3.190299


In [42]:
fangraphs_hitting_fact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/fg_hitting_fact.csv')
fangraphs_pitching_fact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/fangraphs_pitching_fact.csv')

In [43]:
GameBoxScores_fghitting = duckdb.query("""
WITH stats_weighted AS (
    SELECT
        xMLBAMID, Season, AB, PA, Events, 
        BaseRunning,
        (BB_K * PA) as BB_K_W,
        (wRC_plus * PA) as wRC_plus_W,
        (xwOBA * PA) as xwOBA_W,
        (LA * Events) as LA_W
    FROM fangraphs_hitting_fact
)

SELECT GBS.*,
    hb1.HomeB1_BSR_L2, hb1.HomeB1_BB_K_L2, hb1.HomeB1_wRC_plus_L2, hb1.HomeB1_xwOBA_L2, hb1.HomeB1_LA_L2,
    hb2.HomeB2_BSR_L2, hb2.HomeB2_BB_K_L2, hb2.HomeB2_wRC_plus_L2, hb2.HomeB2_xwOBA_L2, hb2.HomeB2_LA_L2,
    hb3.HomeB3_BSR_L2, hb3.HomeB3_BB_K_L2, hb3.HomeB3_wRC_plus_L2, hb3.HomeB3_xwOBA_L2, hb3.HomeB3_LA_L2,
    hb4.HomeB4_BSR_L2, hb4.HomeB4_BB_K_L2, hb4.HomeB4_wRC_plus_L2, hb4.HomeB4_xwOBA_L2, hb4.HomeB4_LA_L2,
    hb5.HomeB5_BSR_L2, hb5.HomeB5_BB_K_L2, hb5.HomeB5_wRC_plus_L2, hb5.HomeB5_xwOBA_L2, hb5.HomeB5_LA_L2,
    hb6.HomeB6_BSR_L2, hb6.HomeB6_BB_K_L2, hb6.HomeB6_wRC_plus_L2, hb6.HomeB6_xwOBA_L2, hb6.HomeB6_LA_L2,
    hb7.HomeB7_BSR_L2, hb7.HomeB7_BB_K_L2, hb7.HomeB7_wRC_plus_L2, hb7.HomeB7_xwOBA_L2, hb7.HomeB7_LA_L2,
    hb8.HomeB8_BSR_L2, hb8.HomeB8_BB_K_L2, hb8.HomeB8_wRC_plus_L2, hb8.HomeB8_xwOBA_L2, hb8.HomeB8_LA_L2,
    hb9.HomeB9_BSR_L2, hb9.HomeB9_BB_K_L2, hb9.HomeB9_wRC_plus_L2, hb9.HomeB9_xwOBA_L2, hb9.HomeB9_LA_L2,
    ab1.AwayB1_BSR_L2, ab1.AwayB1_BB_K_L2, ab1.AwayB1_wRC_plus_L2, ab1.AwayB1_xwOBA_L2, ab1.AwayB1_LA_L2,
    ab2.AwayB2_BSR_L2, ab2.AwayB2_BB_K_L2, ab2.AwayB2_wRC_plus_L2, ab2.AwayB2_xwOBA_L2, ab2.AwayB2_LA_L2,
    ab3.AwayB3_BSR_L2, ab3.AwayB3_BB_K_L2, ab3.AwayB3_wRC_plus_L2, ab3.AwayB3_xwOBA_L2, ab3.AwayB3_LA_L2,
    ab4.AwayB4_BSR_L2, ab4.AwayB4_BB_K_L2, ab4.AwayB4_wRC_plus_L2, ab4.AwayB4_xwOBA_L2, ab4.AwayB4_LA_L2,
    ab5.AwayB5_BSR_L2, ab5.AwayB5_BB_K_L2, ab5.AwayB5_wRC_plus_L2, ab5.AwayB5_xwOBA_L2, ab5.AwayB5_LA_L2,
    ab6.AwayB6_BSR_L2, ab6.AwayB6_BB_K_L2, ab6.AwayB6_wRC_plus_L2, ab6.AwayB6_xwOBA_L2, ab6.AwayB6_LA_L2,
    ab7.AwayB7_BSR_L2, ab7.AwayB7_BB_K_L2, ab7.AwayB7_wRC_plus_L2, ab7.AwayB7_xwOBA_L2, ab7.AwayB7_LA_L2,
    ab8.AwayB8_BSR_L2, ab8.AwayB8_BB_K_L2, ab8.AwayB8_wRC_plus_L2, ab8.AwayB8_xwOBA_L2, ab8.AwayB8_LA_L2,
    ab9.AwayB9_BSR_L2, ab9.AwayB9_BB_K_L2, ab9.AwayB9_wRC_plus_L2, ab9.AwayB9_xwOBA_L2, ab9.AwayB9_LA_L2

FROM GameBoxScores_w_bullpen GBS

-- HOME BATTERS 1-9
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB1_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB1_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB1_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB1_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB1_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter1IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb1 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB2_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB2_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB2_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB2_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB2_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter2IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb2 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB3_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB3_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB3_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB3_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB3_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter3IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb3 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB4_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB4_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB4_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB4_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB4_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter4IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb4 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB5_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB5_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB5_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB5_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB5_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter5IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb5 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB6_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB6_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB6_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB6_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB6_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter6IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb6 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB7_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB7_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB7_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB7_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB7_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter7IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb7 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB8_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB8_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB8_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB8_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB8_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter8IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb8 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB9_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB9_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB9_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB9_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB9_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter9IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb9 ON TRUE

-- AWAY BATTERS 1-9
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB1_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB1_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB1_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB1_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB1_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter1IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab1 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB2_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB2_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB2_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB2_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB2_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter2IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab2 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB3_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB3_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB3_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB3_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB3_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter3IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab3 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB4_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB4_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB4_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB4_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB4_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter4IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab4 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB5_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB5_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB5_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB5_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB5_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter5IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab5 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB6_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB6_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB6_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB6_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB6_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter6IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab6 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB7_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB7_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB7_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB7_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB7_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter7IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab7 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB8_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB8_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB8_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB8_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB8_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter8IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab8 ON TRUE
LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB9_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB9_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB9_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB9_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB9_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter9IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab9 ON TRUE

""").df()

In [44]:
GameBoxScores_fgpitching = duckdb.query("""
WITH stats_weighted AS (
    SELECT
        xMLBAMID,
        Season,
        IP,
        Events,
        WAR,
        (ERA * IP) as ERA_W,
        (K_BB * IP) as K_BB_W,
        (xFIP * IP) as xFIP_W,
        (SIERA * IP) as SIERA_W,
        (GB_pct * Events) as GB_pct_W
    FROM fangraphs_pitching_fact
)

select GBS.*,
    -- Home Pitcher L2 Stats
    hP.HomeSP_WAR_L2,
    hP.HomeSP_ERA_L2,
    hP.HomeSP_K_BB_L2,
    hP.HomeSP_xFIP_L2,
    hP.HomeSP_SIERA_L2,
    hp.HomeSP_GB_pct_L2,
    
    -- Away Pitcher L2 Stats
   ap.AwaySP_WAR_L2,
   ap.AwaySP_ERA_L2,
   ap.AwaySP_K_BB_L2,
   ap.AwaySP_xFIP_L2,
   ap.AwaySP_SIERA_L2,
   ap.AwaySP_GB_pct_L2

FROM GameBoxScores_fghitting GBS

LEFT JOIN LATERAL(
    select 
    SUM(IP) as HomeSP_IP_L2,
    SUM(WAR) as HomeSP_WAR_L2,
    SUM(ERA_W) / NULLIF(SUM(IP), 0) as HomeSP_ERA_L2,
    SUM(K_BB_W) / NULLIF(SUM(IP), 0) as HomeSP_K_BB_L2,
    SUM(xFIP_W) / NULLIF(SUM(IP), 0) as HomeSP_xFIP_L2,
    SUM(SIERA_W) / NULLIF(SUM(IP), 0) as HomeSP_SIERA_L2,
    SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeSP_GB_pct_L2
    FROM(
        SELECT
        IP,
        Events,
        WAR,
        ERA_W,
        K_BB_W,
        xFIP_W,
        SIERA_W,
        GB_pct_W
        from stats_weighted sw
        where sw.xMLBAMID = GBS.HomePitcherIDMLB
        and sw.Season < GBS.Year
        order by sw.Season DESC
        limit 2
    )
) hp on TRUE

LEFT JOIN LATERAL(
   select
   SUM(IP) as AwaySP_IP_L2,
   SUM(WAR) as AwaySP_WAR_L2,
   SUM(ERA_W) / NULLIF(SUM(IP), 0) as AwaySP_ERA_L2,
   SUM(K_BB_W) / NULLIF(SUM(IP), 0) as AwaySP_K_BB_L2,
   SUM(xFIP_W) / NULLIF(SUM(IP), 0) as AwaySP_xFIP_L2,
   SUM(SIERA_W) / NULLIF(SUM(IP), 0) as AwaySP_SIERA_L2,
   SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwaySP_GB_pct_L2
   FROM(
       SELECT
       IP,
       Events,
       WAR,
       ERA_W,
       K_BB_W,
       xFIP_W,
       SIERA_W,
       GB_pct_W
       from stats_weighted sw
       where sw.xMLBAMID = GBS.AwayPitcherIDMLB
       and sw.Season < GBS.Year
       order by sw.Season DESC
       limit 2
   )
) ap on TRUE


""").df()

In [45]:
new_cols = ['DeltaFirstYearCount', 'HomePitcherIDFirstYear', 'AwayPitcherIDFirstYear']

for col in new_cols:
    GameBoxScores_fgpitching[col] = 0

In [46]:
defense_fact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/defense_fact.csv')

In [47]:
defense_fact

,mlb_id,fielding_runs,Month,Year,innings,CompDate
0,425772,0.900000,8,2017,79.111667,2017-08-31
1,425772,0.500000,6,2017,79.111667,2017-06-30
2,425772,0.300000,9,2017,79.111667,2017-09-30
3,455104,0.100000,9,2017,105.611667,2017-09-30
4,455755,0.200000,4,2017,59.111667,2017-04-30
...,...,...,...,...,...,...
22874,660162,-4.439094,9,2018,211.333333,2018-09-30
22875,641531,-4.682033,9,2018,185.666667,2018-09-30
22876,622168,-4.726737,9,2018,164.000000,2018-09-30
22877,595777,-4.759406,9,2018,210.333333,2018-09-30


In [48]:
GameBoxScores = duckdb.query("""
   SELECT
GBS.*,
AC.AC_RUNS AS AwayFieldingRunsC,
A1B.A1B_RUNS AS AwayFieldingRuns1B,
A2B.A2B_RUNS AS AwayFieldingRuns2B,
A3B.A3B_RUNS AS AwayFieldingRuns3B,
ASS.ASS_RUNS AS AwayFieldingRunsSS,
ALF.ALF_RUNS AS AwayFieldingRunsLF,
ACF.ACF_RUNS AS AwayFieldingRunsCF,
ARF.ARF_RUNS AS AwayFieldingRunsRF,
HC.HC_RUNS AS HomeFieldingRunsC,
H1B.H1B_RUNS AS HomeFieldingRuns1B,
H2B.H2B_RUNS AS HomeFieldingRuns2B,
H3B.H3B_RUNS AS HomeFieldingRuns3B,
HSS.HSS_RUNS AS HomeFieldingRunsSS,
HLF.HLF_RUNS AS HomeFieldingRunsLF,
HCF.HCF_RUNS AS HomeFieldingRunsCF,
HRF.HRF_RUNS AS HomeFieldingRunsRF


FROM GameBoxScores_fgpitching GBS


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as AC_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.AwayCIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) AC ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as A1B_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.Away1BIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) A1B ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as A2B_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.Away2BIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) A2B ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as A3B_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.Away3BIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) A3B ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as ASS_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.AwaySSIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) ASS ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as ALF_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.AwayLFIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) ALF ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as ACF_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.AwayCFIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) ACF ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as ARF_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.AwayRFIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) ARF ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as HC_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.HomeCIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) HC ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as H1B_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.Home1BIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) H1B ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as H2B_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.Home2BIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) H2B ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as H3B_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.Home3BIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) H3B ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as HSS_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.HomeSSIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) HSS ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as HLF_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.HomeLFIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) HLF ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as HCF_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.HomeCFIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) HCF ON TRUE


LEFT JOIN LATERAL(
   SELECT SUM(fielding_runs) as HRF_RUNS
   FROM(
       SELECT fielding_runs
       FROM defense_fact df
       WHERE df.mlb_id = GBS.HomeRFIDMLB
       AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
       ORDER BY CAST(df.CompDate as DATE) DESC
       LIMIT 5
   )
) HRF ON TRUE
""").df()



## Clean and Predict Functions (called at the end of daily run function to conduct final cleaning efforts and make model predictions)

In [74]:
def clean1(GameBoxScores):

    stat_list = ['PA', 'BB_K', 'wRC_plus', 'xAVG', 'xSLG']
    removable_cols = []

    for stat in stat_list:
        home_cols = [col for col in GameBoxScores.columns if "L10A" in col and 'Home' in col and f'_{stat}_' in col]
        away_cols = [col for col in GameBoxScores.columns if "L10A" in col and 'Away' in col and f'_{stat}_' in col]
        
        if len(home_cols) != 9 or len(away_cols) != 9:
            print(f"Warning: Found {len(home_cols)} home and {len(away_cols)} away columns for {stat}")

        GameBoxScores[f'Home_Team_{stat}_L10A_Avg'] = GameBoxScores[home_cols].mean(axis=1)
        GameBoxScores[f'Away_Team_{stat}_L10A_Avg'] = GameBoxScores[away_cols].mean(axis=1)
        
        removable_cols.extend(home_cols)
        removable_cols.extend(away_cols)
        
    l5m_stats = ['OPS', 'DRC_plus', 'Kpct']
    groupings = {'Top': [1, 2, 3], 'Middle': [4, 5, 6], 'Bottom': [7, 8, 9]}
    windows = ['L5M']

    for window in windows:
        for stat in l5m_stats:
            for group_name, slots in groupings.items():
                
                home_cols = [f'HomeB{i}_{stat}_{window}' for i in slots]
                away_cols = [f'AwayB{i}_{stat}_{window}' for i in slots]
                
                if all(col in GameBoxScores.columns for col in home_cols + away_cols):
                    new_col_home = f'Home_{group_name}_{stat}_{window}_Avg'
                    new_col_away = f'Away_{group_name}_{stat}_{window}_Avg'
                    
                    GameBoxScores[new_col_home] = GameBoxScores[home_cols].mean(axis=1)
                    GameBoxScores[new_col_away] = GameBoxScores[away_cols].mean(axis=1)
                    
                    removable_cols.extend(home_cols)
                    removable_cols.extend(away_cols)
                else:
                    # Debugging print to see which specific columns are missing
                    missing = [c for c in home_cols + away_cols if c not in GameBoxScores.columns]
                    
    l5m_stats = ['OPS', 'BBpct']
    groupings = {'Top': [1, 2, 3], 'Middle': [4, 5, 6], 'Bottom': [7, 8, 9]}
    windows = ['L5M_P']

    for window in windows:
        for stat in l5m_stats:
            for group_name, slots in groupings.items():
                
                home_cols = [f'HomeB{i}_{stat}_{window}' for i in slots]
                away_cols = [f'AwayB{i}_{stat}_{window}' for i in slots]
                
                if all(col in GameBoxScores.columns for col in home_cols + away_cols):
                    new_col_home = f'Home_{group_name}_{stat}_{window}_Avg'
                    new_col_away = f'Away_{group_name}_{stat}_{window}_Avg'
                    
                    GameBoxScores[new_col_home] = GameBoxScores[home_cols].mean(axis=1)
                    GameBoxScores[new_col_away] = GameBoxScores[away_cols].mean(axis=1)
                    
                    removable_cols.extend(home_cols)
                    removable_cols.extend(away_cols)
                else:
                    # Debugging print to see which specific columns are missing
                    missing = [c for c in home_cols + away_cols if c not in GameBoxScores.columns]

    GameBoxScores = GameBoxScores.drop(columns=removable_cols, errors='ignore')

    remove = ['AwayPitcherID', 'AwayPitcher', 'HomePitcherID', 'HomePitcher', 'AwayBatter1ID', 'AwayBatter1', 'AwayBatter1Pos', 'AwayBatter2ID', 'AwayBatter2', 'AwayBatter2Pos', 'AwayBatter3ID', 'AwayBatter3',
            'AwayBatter3Pos', 'AwayBatter4ID', 'AwayBatter4', 'AwayBatter4Pos', 'AwayBatter5ID', 'AwayBatter5', 'AwayBatter5Pos', 'AwayBatter6ID', 'AwayBatter6', 'AwayBatter6Pos', 'AwayBatter7ID', 'AwayBatter7',
            'AwayBatter7Pos', 'AwayBatter8ID', 'AwayBatter8', 'AwayBatter8Pos', 'AwayBatter9ID', 'AwayBatter9', 'AwayBatter9Pos', 'HomeBatter1ID', 'HomeBatter1', 'HomeBatter1Pos', 'HomeBatter2ID', 'HomeBatter2',
            'HomeBatter2Pos', 'HomeBatter3ID', 'HomeBatter3', 'HomeBatter3Pos', 'HomeBatter4ID', 'HomeBatter4', 'HomeBatter4Pos', 'HomeBatter5ID', 'HomeBatter5', 'HomeBatter5Pos', 'HomeBatter6ID', 'HomeBatter6',
            'HomeBatter6Pos', 'HomeBatter7ID', 'HomeBatter7', 'HomeBatter7Pos', 'HomeBatter8ID', 'HomeBatter8', 'HomeBatter8Pos', 'HomeBatter9ID', 'HomeBatter9', 'HomeBatter9Pos','AwayPitcherIDMLB',
            'AwayPitcherIDBRef', 'AwayPitcherIDFG', 'HomePitcherIDMLB', 'HomePitcherIDBRef', 'HomePitcherIDFG', 'AwayBatter1IDMLB', 'AwayBatter1IDBRef', 'AwayBatter1IDFG', 'AwayBatter2IDMLB', 'AwayBatter2IDBRef',
            'AwayBatter2IDFG', 'AwayBatter3IDMLB', 'AwayBatter3IDBRef', 'AwayBatter3IDFG', 'AwayBatter4IDMLB', 'AwayBatter4IDBRef', 'AwayBatter4IDFG', 'AwayBatter5IDMLB', 'AwayBatter5IDBRef', 'AwayBatter5IDFG',
            'AwayBatter6IDMLB', 'AwayBatter6IDBRef', 'AwayBatter6IDFG', 'AwayBatter7IDMLB', 'AwayBatter7IDBRef', 'AwayBatter7IDFG', 'AwayBatter8IDMLB', 'AwayBatter8IDBRef', 'AwayBatter8IDFG', 'AwayBatter9IDMLB',
            'AwayBatter9IDBRef', 'AwayBatter9IDFG', 'HomeBatter1IDMLB', 'HomeBatter1IDBRef', 'HomeBatter1IDFG', 'HomeBatter2IDMLB', 'HomeBatter2IDBRef', 'HomeBatter2IDFG', 'HomeBatter3IDMLB', 'HomeBatter3IDBRef',
            'HomeBatter3IDFG', 'HomeBatter4IDMLB', 'HomeBatter4IDBRef', 'HomeBatter4IDFG', 'HomeBatter5IDMLB', 'HomeBatter5IDBRef', 'HomeBatter5IDFG', 'HomeBatter6IDMLB', 'HomeBatter6IDBRef', 'HomeBatter6IDFG',
            'HomeBatter7IDMLB', 'HomeBatter7IDBRef', 'HomeBatter7IDFG', 'HomeBatter8IDMLB', 'HomeBatter8IDBRef', 'HomeBatter8IDFG', 'HomeBatter9IDMLB', 'HomeBatter9IDBRef', 'HomeBatter9IDFG','AwayBatter1IDFirstYear',
            'AwayBatter2IDFirstYear', 'AwayBatter3IDFirstYear', 'AwayBatter4IDFirstYear', 'AwayBatter5IDFirstYear', 'AwayBatter6IDFirstYear', 'AwayBatter7IDFirstYear', 'AwayBatter8IDFirstYear',
            'AwayBatter9IDFirstYear', 'HomeBatter1IDFirstYear', 'HomeBatter2IDFirstYear', 'HomeBatter3IDFirstYear', 'HomeBatter4IDFirstYear', 'HomeBatter5IDFirstYear', 'HomeBatter6IDFirstYear', 
            'HomeBatter7IDFirstYear', 'HomeBatter8IDFirstYear', 'HomeBatter9IDFirstYear', 'AwayCIDMLB', 'Away1BIDMLB', 'Away2BIDMLB', 'Away3BIDMLB', 'AwaySSIDMLB', 'AwayLFIDMLB', 'AwayCFIDMLB',
            'AwayRFIDMLB', 'HomeCIDMLB', 'Home1BIDMLB', 'Home2BIDMLB', 'Home3BIDMLB', 'HomeSSIDMLB', 'HomeLFIDMLB', 'HomeCFIDMLB', 'HomeRFIDMLB']

    GameBoxScores = GameBoxScores.drop(columns=remove, errors='ignore')

    # Define the position mappings
    infield_pos = [ '1B', '2B', '3B', 'SS']
    outfield_pos = ['LF', 'CF', 'RF']

    for side in ['Home', 'Away']:
        if_cols = [f'{side}FieldingRuns{pos}' for pos in infield_pos]
        of_cols = [f'{side}FieldingRuns{pos}' for pos in outfield_pos]
        
        GameBoxScores[f'{side}_Infield_Defense_L5M'] = GameBoxScores[if_cols].sum(axis=1)
        GameBoxScores[f'{side}_Outfield_Defense_L5M'] = GameBoxScores[of_cols].sum(axis=1)
        
        removable_cols.extend(if_cols)
        removable_cols.extend(of_cols)
        
        GameBoxScores = GameBoxScores.drop(columns=removable_cols, errors='ignore')
        
    return GameBoxScores
    
    

In [75]:
def clean2(GameBoxScores):
    
    GameBoxScores['HomePitcherIDFirstYear'] = 0
    GameBoxScores['AwayPitcherIDFirstYear'] = 0
    GameBoxScores['DeltaFirstYearCount'] = 0

    delta_list = ['HomeFieldingRunsC', 'HomeSP_IP_L2', 'HomeSP_WAR_L2', 'HomeSP_ERA_L2', 'HomeSP_K_9_L2', 'HomeSP_BB_9_L2', 'HomeSP_K_BB_L2', 'HomeSP_FIP_L2', 'HomeSP_xFIP_L2', 'HomeSP_SIERA_L2', 'HomeSP_kwERA_L2',
                'HomeSP_xERA_L2', 'HomeSP_Barrel_pct_L2', 'HomeSP_GB_FB_L2', 'HomeSP_GB_pct_L2', 'HomeSP_FB_pct_L2', 'HomeSP_IP_L5M', 'HomeSP_ERA_L5M', 'HomeSP_DRA_L5M', 'HomeSP_Kpct_L5M', 'HomeSP_BBpct_L5M',
                'HomeBP_IP_L3M', 'HomeBP_ERA_L3M', 'HomeBP_SO9_L3M', 'HomeBP_BB9_L3M', 'HomeSP_IP_L4A', 'HomeSP_ERA_L4A', 'HomeSP_SO9_L4A', 'HomeSP_BB9_L4A', 'HomeSP_K_BB_L4A', 'HomeSP_FIP_L4A', 'HomeSP_xFIP_L4A',
                'HomeSP_SIERA_L4A', 'HomeSP_xERA_L4A', 'HomeSP_Barrel_pct_L4A', 'HomeSP_GB_FB_L4A', 'HomeSP_GB_pct_L4A', 'HomeSP_FB_pct_L4A', 'HomeB1_PA_L2', 'HomeB1_WAR_L2', 'HomeB1_BSR_L2', 'HomeB1_BB_pct_L2',
                'HomeB1_K_pct_L2', 'HomeB1_BB_K_L2', 'HomeB1_OPS_L2', 'HomeB1_GB_FB_L2', 'HomeB1_GB_pct_L2', 'HomeB1_FB_pct_L2', 'HomeB1_wOBA_L2', 'HomeB1_wRC_plus_L2', 'HomeB1_xwOBA_L2', 'HomeB1_xAVG_L2', 'HomeB1_xSLG_L2',
                'HomeB1_LA_L2', 'HomeB1_Barrel_pct_L2', 'HomeB1_maxEV_L2', 'HomeB1_EV90_L2', 'HomeB2_PA_L2', 'HomeB2_WAR_L2', 'HomeB2_BSR_L2', 'HomeB2_BB_pct_L2', 'HomeB2_K_pct_L2', 'HomeB2_BB_K_L2', 'HomeB2_OPS_L2',
                'HomeB2_GB_FB_L2', 'HomeB2_GB_pct_L2', 'HomeB2_FB_pct_L2', 'HomeB2_wOBA_L2', 'HomeB2_wRC_plus_L2', 'HomeB2_xwOBA_L2', 'HomeB2_xAVG_L2', 'HomeB2_xSLG_L2', 'HomeB2_LA_L2', 'HomeB2_Barrel_pct_L2', 
                'HomeB2_maxEV_L2', 'HomeB2_EV90_L2', 'HomeB3_PA_L2', 'HomeB3_WAR_L2', 'HomeB3_BSR_L2', 'HomeB3_BB_pct_L2', 'HomeB3_K_pct_L2', 'HomeB3_BB_K_L2', 'HomeB3_OPS_L2', 'HomeB3_GB_FB_L2', 'HomeB3_GB_pct_L2',
                'HomeB3_FB_pct_L2', 'HomeB3_wOBA_L2', 'HomeB3_wRC_plus_L2', 'HomeB3_xwOBA_L2', 'HomeB3_xAVG_L2', 'HomeB3_xSLG_L2', 'HomeB3_LA_L2', 'HomeB3_Barrel_pct_L2', 'HomeB3_maxEV_L2', 'HomeB3_EV90_L2', 'HomeB4_PA_L2',
                'HomeB4_WAR_L2', 'HomeB4_BSR_L2', 'HomeB4_BB_pct_L2', 'HomeB4_K_pct_L2', 'HomeB4_BB_K_L2', 'HomeB4_OPS_L2', 'HomeB4_GB_FB_L2', 'HomeB4_GB_pct_L2', 'HomeB4_FB_pct_L2', 'HomeB4_wOBA_L2', 'HomeB4_wRC_plus_L2',
                'HomeB4_xwOBA_L2', 'HomeB4_xAVG_L2', 'HomeB4_xSLG_L2', 'HomeB4_LA_L2', 'HomeB4_Barrel_pct_L2', 'HomeB4_maxEV_L2', 'HomeB4_EV90_L2', 'HomeB5_PA_L2', 'HomeB5_WAR_L2', 'HomeB5_BSR_L2', 'HomeB5_BB_pct_L2',
                'HomeB5_K_pct_L2', 'HomeB5_BB_K_L2', 'HomeB5_OPS_L2', 'HomeB5_GB_FB_L2', 'HomeB5_GB_pct_L2', 'HomeB5_FB_pct_L2', 'HomeB5_wOBA_L2', 'HomeB5_wRC_plus_L2', 'HomeB5_xwOBA_L2', 'HomeB5_xAVG_L2', 'HomeB5_xSLG_L2',
                'HomeB5_LA_L2', 'HomeB5_Barrel_pct_L2', 'HomeB5_maxEV_L2', 'HomeB5_EV90_L2', 'HomeB6_PA_L2', 'HomeB6_WAR_L2', 'HomeB6_BSR_L2', 'HomeB6_BB_pct_L2', 'HomeB6_K_pct_L2', 'HomeB6_BB_K_L2', 'HomeB6_OPS_L2',
                'HomeB6_GB_FB_L2', 'HomeB6_GB_pct_L2', 'HomeB6_FB_pct_L2', 'HomeB6_wOBA_L2', 'HomeB6_wRC_plus_L2', 'HomeB6_xwOBA_L2', 'HomeB6_xAVG_L2', 'HomeB6_xSLG_L2', 'HomeB6_LA_L2', 'HomeB6_Barrel_pct_L2', 
                'HomeB6_maxEV_L2', 'HomeB6_EV90_L2', 'HomeB7_PA_L2', 'HomeB7_WAR_L2', 'HomeB7_BSR_L2', 'HomeB7_BB_pct_L2', 'HomeB7_K_pct_L2', 'HomeB7_BB_K_L2', 'HomeB7_OPS_L2', 'HomeB7_GB_FB_L2', 'HomeB7_GB_pct_L2',
                'HomeB7_FB_pct_L2', 'HomeB7_wOBA_L2', 'HomeB7_wRC_plus_L2', 'HomeB7_xwOBA_L2', 'HomeB7_xAVG_L2', 'HomeB7_xSLG_L2', 'HomeB7_LA_L2', 'HomeB7_Barrel_pct_L2', 'HomeB7_maxEV_L2', 'HomeB7_EV90_L2',
                'HomeB8_PA_L2', 'HomeB8_WAR_L2', 'HomeB8_BSR_L2', 'HomeB8_BB_pct_L2', 'HomeB8_K_pct_L2', 'HomeB8_BB_K_L2', 'HomeB8_OPS_L2', 'HomeB8_GB_FB_L2', 'HomeB8_GB_pct_L2', 'HomeB8_FB_pct_L2', 'HomeB8_wOBA_L2',
                'HomeB8_wRC_plus_L2', 'HomeB8_xwOBA_L2', 'HomeB8_xAVG_L2', 'HomeB8_xSLG_L2', 'HomeB8_LA_L2', 'HomeB8_Barrel_pct_L2', 'HomeB8_maxEV_L2', 'HomeB8_EV90_L2', 'HomeB9_PA_L2', 'HomeB9_WAR_L2', 'HomeB9_BSR_L2',
                'HomeB9_BB_pct_L2', 'HomeB9_K_pct_L2', 'HomeB9_BB_K_L2', 'HomeB9_OPS_L2', 'HomeB9_GB_FB_L2', 'HomeB9_GB_pct_L2', 'HomeB9_FB_pct_L2', 'HomeB9_wOBA_L2', 'HomeB9_wRC_plus_L2', 'HomeB9_xwOBA_L2', 
                'HomeB9_xAVG_L2', 'HomeB9_xSLG_L2', 'HomeB9_LA_L2', 'HomeB9_Barrel_pct_L2', 'HomeB9_maxEV_L2', 'HomeB9_EV90_L2', 'HomeTeamWinsL162', 'HomeTeamWinsL50', 'HomeTeamWinsL10', 'HomeTeamRunsScoredL10',
                'HomeTeamRunsAllowedL10', 'HomeTeamRunsScoredL100', 'HomeTeamRunsAllowedL100', 'HomeTeamAvgOutTotalL3', 'Home_Team_PA_L10A_Avg', 'Home_Team_AVG_L10A_Avg', 'Home_Team_BB_pct_L10A_Avg', 
                'Home_Team_K_pct_L10A_Avg', 'Home_Team_BB_K_L10A_Avg', 'Home_Team_OBP_L10A_Avg', 'Home_Team_OPS_L10A_Avg', 'Home_Team_GB_FB_L10A_Avg', 'Home_Team_wOBA_L10A_Avg', 'Home_Team_wRC_plus_L10A_Avg',
                'Home_Team_xwOBA_L10A_Avg', 'Home_Team_xAVG_L10A_Avg', 'Home_Team_xSLG_L10A_Avg', 'Home_Team_Barrel_pct_L10A_Avg', 'Home_Top_PA_L5M_Avg', 'Home_Middle_PA_L5M_Avg', 'Home_Bottom_PA_L5M_Avg', 
                'Home_Top_OBP_L5M_Avg', 'Home_Middle_OBP_L5M_Avg', 'Home_Bottom_OBP_L5M_Avg', 'Home_Top_OPS_L5M_Avg', 'Home_Middle_OPS_L5M_Avg', 'Home_Bottom_OPS_L5M_Avg', 'Home_Top_DRC_plus_L5M_Avg',
                'Home_Middle_DRC_plus_L5M_Avg', 'Home_Bottom_DRC_plus_L5M_Avg', 'Home_Top_Kpct_L5M_Avg', 'Home_Middle_Kpct_L5M_Avg', 'Home_Bottom_Kpct_L5M_Avg', 'Home_Top_BBpct_L5M_Avg', 'Home_Middle_BBpct_L5M_Avg',
                'Home_Bottom_BBpct_L5M_Avg', 'Home_Top_PA_L5M_P_Avg', 'Home_Middle_PA_L5M_P_Avg', 'Home_Bottom_PA_L5M_P_Avg', 'Home_Top_OBP_L5M_P_Avg', 'Home_Middle_OBP_L5M_P_Avg', 'Home_Bottom_OBP_L5M_P_Avg',
                'Home_Top_OPS_L5M_P_Avg', 'Home_Middle_OPS_L5M_P_Avg', 'Home_Bottom_OPS_L5M_P_Avg', 'Home_Top_DRC_plus_L5M_P_Avg', 'Home_Middle_DRC_plus_L5M_P_Avg', 'Home_Bottom_DRC_plus_L5M_P_Avg', 
                'Home_Top_Kpct_L5M_P_Avg', 'Home_Middle_Kpct_L5M_P_Avg', 'Home_Bottom_Kpct_L5M_P_Avg', 'Home_Top_BBpct_L5M_P_Avg', 'Home_Middle_BBpct_L5M_P_Avg', 'Home_Bottom_BBpct_L5M_P_Avg', 'HomeFirstYearCount',
                'Home_Infield_Defense_L5M', 'Home_Outfield_Defense_L5M']

    for col in delta_list:
        away_col = col.replace("Home", "Away")
        delta_col = col.replace("Home", "Delta") 
        
        if col in GameBoxScores.columns and away_col in GameBoxScores.columns:
            GameBoxScores[delta_col] = GameBoxScores[col] - GameBoxScores[away_col]

    col_removal = []
    for col in delta_list:
        away_col = col.replace("Home", "Away")
        delta_col = col.replace("Home", "Delta") 
        
        if away_col in GameBoxScores.columns:
            col_removal.append(away_col)
        elif col in GameBoxScores.columns:
            col_removal.append(col)
        
    GameBoxScores.drop(columns=col_removal, inplace=True)

    keep_exceptions = ['HomePitcherIDFirstYear', 'AwayPitcherIDFirstYear', 'HomeTeam', 'AwayTeam']

    # Filter columns
    cols_to_keep = [
        col for col in GameBoxScores.columns 
        if not (col.startswith('Home') or col.startswith('Away')) 
        or col in keep_exceptions
    ]

    # Reassign the filtered DataFrame
    GameBoxScores = GameBoxScores[cols_to_keep]

    numeric_cols = GameBoxScores.select_dtypes(include=['number']).columns

    # Fill missing values in numeric columns with their respective means
    GameBoxScores[numeric_cols] = GameBoxScores[numeric_cols].fillna(GameBoxScores[numeric_cols].mean())
    
    return GameBoxScores



In [80]:
def predict_records(GameBoxScores):

    x_g_cols = ['DeltaTeamWinsL162', 'DeltaTeamWinsL50', 'DeltaTeamRunsScoredL100', 'DeltaTeamRunsAllowedL100', 
        'DeltaFirstYearCount', 'HomePitcherIDFirstYear', 'AwayPitcherIDFirstYear', 'Delta_Infield_Defense_L5M', 
        'Delta_Outfield_Defense_L5M', 'DeltaFieldingRunsC', 'DeltaSP_xFIP_L2', 'DeltaSP_SIERA_L2', 
        'DeltaSP_ERA_L2', 'DeltaSP_WAR_L2', 'DeltaSP_K_BB_L2', 'DeltaSP_GB_pct_L2', 'DeltaSP_DRA_L5M', 
        'DeltaSP_Kpct_L5M', 'DeltaSP_BBpct_L5M', 'DeltaSP_IP_L5M', 'DeltaSP_IP_L4A', 'DeltaSP_xERA_L4A', 
        'DeltaSP_Barrel_pct_L4A', 'DeltaSP_K_BB_L4A', 'Delta_Team_wRC_plus_L10A_Avg', 'Delta_Team_PA_L10A_Avg', 
        'Delta_Team_xAVG_L10A_Avg', 'Delta_Team_xSLG_L10A_Avg', 'Delta_Team_BB_K_L10A_Avg', 
        'Delta_Top_DRC_plus_L5M_Avg', 'Delta_Middle_DRC_plus_L5M_Avg', 'Delta_Bottom_DRC_plus_L5M_Avg', 
        'Delta_Top_OPS_L5M_Avg', 'Delta_Middle_OPS_L5M_Avg', 'Delta_Bottom_OPS_L5M_Avg', 
        'Delta_Top_OPS_L5M_P_Avg', 'Delta_Middle_OPS_L5M_P_Avg', 'Delta_Bottom_OPS_L5M_P_Avg', 
        'Delta_Top_Kpct_L5M_Avg', 'Delta_Middle_Kpct_L5M_Avg', 'Delta_Bottom_Kpct_L5M_Avg', 
        'Delta_Top_BBpct_L5M_P_Avg', 'Delta_Middle_BBpct_L5M_P_Avg', 'Delta_Bottom_BBpct_L5M_P_Avg', 
        'DeltaB1_wRC_plus_L2', 'DeltaB2_wRC_plus_L2', 'DeltaB3_wRC_plus_L2', 'DeltaB4_wRC_plus_L2', 
        'DeltaB5_wRC_plus_L2', 'DeltaB6_wRC_plus_L2', 'DeltaB7_wRC_plus_L2', 'DeltaB8_wRC_plus_L2', 
        'DeltaB9_wRC_plus_L2', 'DeltaB1_xwOBA_L2', 'DeltaB2_xwOBA_L2', 'DeltaB3_xwOBA_L2', 
        'DeltaB4_xwOBA_L2', 'DeltaB5_xwOBA_L2', 'DeltaB6_xwOBA_L2', 'DeltaB7_xwOBA_L2', 
        'DeltaB8_xwOBA_L2', 'DeltaB9_xwOBA_L2', 'DeltaB1_BB_K_L2', 'DeltaB2_BB_K_L2', 
        'DeltaB3_BB_K_L2', 'DeltaB4_BB_K_L2', 'DeltaB5_BB_K_L2', 'DeltaB6_BB_K_L2', 
        'DeltaB7_BB_K_L2', 'DeltaB8_BB_K_L2', 'DeltaB9_BB_K_L2', 'DeltaB1_LA_L2', 
        'DeltaB2_LA_L2', 'DeltaB3_LA_L2', 'DeltaB4_LA_L2', 'DeltaB5_LA_L2', 'DeltaB6_LA_L2', 
        'DeltaB7_LA_L2', 'DeltaB8_LA_L2', 'DeltaB9_LA_L2', 'DeltaB1_BSR_L2', 'DeltaB2_BSR_L2', 
        'DeltaB3_BSR_L2', 'DeltaB4_BSR_L2', 'DeltaB5_BSR_L2', 'DeltaB6_BSR_L2', 'DeltaB7_BSR_L2', 
        'DeltaB8_BSR_L2', 'DeltaB9_BSR_L2', 'DeltaBP_SO9_L3M', 'DeltaBP_BB9_L3M', 'DeltaBP_ERA_L3M'
    ]

    import joblib
    import tensorflow as tf
    
    # Load Scaler and non-Keras models
    scaler = joblib.load('game_data_scaler.joblib')
    rf_model = joblib.load('best_rf_model.joblib')

    # Load the Keras Neural Network
    nn_model = tf.keras.models.load_model('brewers_best_nn4_model.keras')

    # Prepare and Scale Features
    X_live = GameBoxScores[x_g_cols]
    X_scaled = scaler.transform(X_live)

    all_preds = []

    # Get predictions from sklearn/gradient boosted models

    all_preds.append(rf_model.predict_proba(X_scaled)[:, 1])

    # Get predictions from Neural Network
    nn_preds = nn_model.predict(X_scaled).flatten()
    all_preds.append(nn_preds)

    # Final Ensemble Average
    ensemble_df = pd.DataFrame(all_preds).T
    GameBoxScores['final_prediction'] = ensemble_df.mean(axis=1)
    GameBoxScores['inverse_prediction'] = 1 - GameBoxScores['final_prediction']
    
    return GameBoxScores

# Activation Function

### This function is what will be run on a daily basis to create the predictions for a given day. 

### It is essentially some data loading and a massive string of SQL statements. The clean1, clean2, and predict_records fucntions are called at the end

In [93]:
def daily_run():
    player_dim = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/player_dim_live.csv')
    SPGameLogFact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/SPGameLogFactCombined.csv')
    BatterGameLogFact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/Batter_Combined_Fact.csv')
    dra_map_clean = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/dra_map_clean.csv')
    drc_map_clean = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/drc_map_clean.csv')
    drcp_map_clean = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/drcp_map_clean.csv')
    team_bullpen_monthly = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/team_bullpen_monthly.csv')
    team_bullpen_monthly['CompDate'] = (
        pd.to_datetime(team_bullpen_monthly['query_year'].astype(str) + '-' + team_bullpen_monthly['query_month'].astype(str) + '-01')
        + pd.offsets.MonthEnd(0)
    )
    fangraphs_hitting_fact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/fg_hitting_fact.csv')
    fangraphs_pitching_fact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/fangraphs_pitching_fact.csv')
    defense_fact = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/defense_fact.csv')
    
    
    t_lineups = get_daily_lineups1()
    t_pitchers = get_daily_pitchers()
    t_matchups = get_matchups()
    
    abbr_conversion = {
        'WSH': 'WAS',
        'SD': 'SDP',
        'CWS': 'CHW',
        'SF': 'SFG',
        'TB': 'TBR',
        'KC': 'KCR'   
    }
    t_lineups['order'] = range(1, len(t_lineups) + 1)
    batting_order = [1, 2, 3, 4, 5, 6, 7, 8, 9]
    t_lineups['battingSpot'] = (batting_order * (len(t_lineups) // 9 + 1))[:len(t_lineups)]
    t_lineups['Team'] = t_lineups['Team'].replace(abbr_conversion)
    t_pitchers['Team'] = t_pitchers['Team'].replace(abbr_conversion)
    t_matchups['HomeTeam'] = t_matchups['HomeTeam'].replace(abbr_conversion)
    t_matchups['AwayTeam'] = t_matchups['AwayTeam'].replace(abbr_conversion)
    
    t_lineups['Name'] = t_lineups['Name'].str.lower()
    t_pitchers['Name'] = t_pitchers['Name'].str.lower()
    
    player_dim = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/player_dim_live.csv')
    
    t_lineups = duckdb.query("""
        WITH name_counts AS (
            -- Step 1: Identify which names only appear once in player_dim
            SELECT player_name, COUNT(*) as name_occurrence
            FROM player_dim
            GROUP BY player_name
        )
        SELECT 
            tl.*,
            pd.fg_id, 
            pd.mlb_id
        FROM t_lineups tl
        LEFT JOIN player_dim pd 
            ON tl.Name = pd.player_name
        LEFT JOIN name_counts nc
            ON tl.Name = nc.player_name
        WHERE 
            -- Rule 1: It's a match if the teams align
            tl.Team = pd.Team 
            OR 
            -- Rule 2: It's a match if that name only exists once in the whole dimension table
            nc.name_occurrence = 1
            OR
            -- Rule 3: Catch-all if we haven't assigned a team yet but need the record
            pd.Team IS NULL
        
        -- Step 2: Use QUALIFY to keep exactly ONE record per original lineup slot
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY tl.order 
            ORDER BY (tl.Team = pd.Team) DESC, pd.fg_id ASC
        ) = 1
        
        ORDER BY tl.order
    """).df()
    
    t_pitchers = duckdb.query("""
        WITH name_counts AS (
            SELECT player_name, COUNT(*) as name_occurrence
            FROM player_dim
            GROUP BY player_name
        )
        SELECT 
            tl.*,
            pd.fg_id, 
            pd.mlb_id
        FROM t_pitchers tl
        LEFT JOIN player_dim pd 
            ON tl.Name = pd.player_name
        LEFT JOIN name_counts nc
            ON tl.Name = nc.player_name
        WHERE 
            tl.Team = pd.Team 
            OR 
            nc.name_occurrence = 1
            OR
            pd.Team IS NULL
        
        -- FIX: Changed 'tl.order' to 'tl.Name'
        QUALIFY ROW_NUMBER() OVER (
            PARTITION BY tl.Name 
            ORDER BY (tl.Team = pd.Team) DESC, pd.fg_id ASC
        ) = 1
    """).df()
    
    gbs_base = duckdb.query("""
        SELECT 
            m.HomeTeam,
            m.AwayTeam,
            -- Home Pitcher Data
            hp.Name as HomePitcherName,
            hp.Hand as HomePitcherHand,
            hp.fg_id as HomePitcherIDFG,
            hp.mlb_id as HomePitcherIDMLB,
            -- Away Pitcher Data
            ap.Name as AwayPitcherName,
            ap.Hand as AwayPitcherHand,
            ap.fg_id as AwayPitcherIDFG,
            ap.mlb_id as AwayPitcherIDMLB
        FROM t_matchups m
        -- Join for the Home Pitcher
        LEFT JOIN t_pitchers hp 
            ON m.HomeTeam = hp.Team
        -- Join for the Away Pitcher
        LEFT JOIN t_pitchers ap 
            ON m.AwayTeam = ap.Team
    """).df()
    
    lineups_wide = duckdb.query("""
    WITH teams AS (
        SELECT DISTINCT Team FROM todays_lineups
    )
    SELECT 
        t.Team,
        -- Names
        b1.Name AS B1Name, b2.Name AS B2Name, b3.Name AS B3Name, 
        b4.Name AS B4Name, b5.Name AS B5Name, b6.Name AS B6Name, 
        b7.Name AS B7Name, b8.Name AS B8Name, b9.Name AS B9Name,
        
        -- FG IDs
        b1.fg_id AS B1IDFG, b2.fg_id AS B2IDFG, b3.fg_id AS B3IDFG, 
        b4.fg_id AS B4IDFG, b5.fg_id AS B5IDFG, b6.fg_id AS B6IDFG, 
        b7.fg_id AS B7IDFG, b8.fg_id AS B8IDFG, b9.fg_id AS B9IDFG,
        
        -- MLB IDs
        b1.mlb_id AS B1IDMLB, b2.mlb_id AS B2IDMLB, b3.mlb_id AS B3IDMLB, 
        b4.mlb_id AS B4IDMLB, b5.mlb_id AS B5IDMLB, b6.mlb_id AS B6IDMLB, 
        b7.mlb_id AS B7IDMLB, b8.mlb_id AS B8IDMLB, b9.mlb_id AS B9IDMLB,
        
        -- Positional IDs (Defensive)
        p1b.mlb_id AS IDMLB_1B, p2b.mlb_id AS IDMLB_2B, pss.mlb_id AS IDMLB_SS,
        p3b.mlb_id AS IDMLB_3B, pc.mlb_id  AS IDMLB_C,  plf.mlb_id AS IDMLB_LF,
        pcf.mlb_id AS IDMLB_CF, prf.mlb_id AS IDMLB_RF
        
        FROM teams t
        -- Batting Order Joins
        LEFT JOIN t_lineups b1 ON t.Team = b1.Team AND CAST(b1.battingSpot AS INT) = 1
        LEFT JOIN t_lineups b2 ON t.Team = b2.Team AND CAST(b2.battingSpot AS INT) = 2
        LEFT JOIN t_lineups b3 ON t.Team = b3.Team AND CAST(b3.battingSpot AS INT) = 3
        LEFT JOIN t_lineups b4 ON t.Team = b4.Team AND CAST(b4.battingSpot AS INT) = 4
        LEFT JOIN t_lineups b5 ON t.Team = b5.Team AND CAST(b5.battingSpot AS INT) = 5
        LEFT JOIN t_lineups b6 ON t.Team = b6.Team AND CAST(b6.battingSpot AS INT) = 6
        LEFT JOIN t_lineups b7 ON t.Team = b7.Team AND CAST(b7.battingSpot AS INT) = 7
        LEFT JOIN t_lineups b8 ON t.Team = b8.Team AND CAST(b8.battingSpot AS INT) = 8
        LEFT JOIN t_lineups b9 ON t.Team = b9.Team AND CAST(b9.battingSpot AS INT) = 9
        
        -- Defensive Position Joins
        LEFT JOIN t_lineups p1b ON t.Team = p1b.Team AND p1b.Position = '1B'
        LEFT JOIN t_lineups p2b ON t.Team = p2b.Team AND p2b.Position = '2B'
        LEFT JOIN t_lineups pss ON t.Team = pss.Team AND pss.Position = 'SS'
        LEFT JOIN t_lineups p3b ON t.Team = p3b.Team AND p3b.Position = '3B'
        LEFT JOIN t_lineups pc  ON t.Team = pc.Team  AND pc.Position = 'C'
        LEFT JOIN t_lineups plf ON t.Team = plf.Team AND plf.Position = 'LF'
        LEFT JOIN t_lineups pcf ON t.Team = pcf.Team AND pcf.Position = 'CF'
        LEFT JOIN t_lineups prf ON t.Team = prf.Team AND prf.Position = 'RF'
    """).df()

    gbs_pre_stats = duckdb.query("""
        SELECT 
            g.*,
            -- Home Lineup (Aliases added for clarity)
            hl.B1Name AS HomeB1Name, hl.B2Name AS HomeB2Name, hl.B3Name AS HomeB3Name, 
            hl.B4Name AS HomeB4Name, hl.B5Name AS HomeB5Name, hl.B6Name AS HomeB6Name, 
            hl.B7Name AS HomeB7Name, hl.B8Name AS HomeB8Name, hl.B9Name AS HomeB9Name,
            hl.B1IDFG AS HomeBatter1IDFG, hl.B2IDFG AS HomeBatter2IDFG, hl.B3IDFG AS HomeBatter3IDFG, hl.B4IDFG AS HomeBatter4IDFG, hl.B5IDFG as HomeBatter5IDFG,
            hl.B6IDFG AS HomeBatter6IDFG, hl.B7IDFG AS HomeBatter7IDFG, hl.B8IDFG AS HomeBatter8IDFG, hl.B9IDFG AS HomeBatter9IDFG,
            hl.B1IDMLB AS HomeBatter1IDMLB, hl.B2IDMLB AS HomeBatter2IDMLB, hl.B3IDMLB AS HomeBatter3IDMLB, hl.B4IDMLB AS HomeBatter4IDMLB, hl.B5IDMLB AS HomeBatter5IDMLB,
            hl.B6IDMLB AS HomeBatter6IDMLB, hl.B7IDMLB AS HomeBatter7IDMLB, hl.B8IDMLB AS HomeBatter8IDMLB, hl.B9IDMLB AS HomeBatter9IDMLB,
            hl.IDMLB_1B AS Home1BIDMLB, hl.IDMLB_2B AS Home2BIDMLB, hl.IDMLB_SS AS HomeSSIDMLB,
            hl.IDMLB_3B AS Home3BIDMLB, hl.IDMLB_C AS HomeCIDMLB, hl.IDMLB_LF AS HomeLFIDMLB,
            hl.IDMLB_CF AS HomeCFIDMLB, hl.IDMLB_RF AS HomeRFIDMLB,
            
            -- Away Lineup
        al.B1Name AS AwayB1Name, al.B2Name AS AwayB2Name, al.B3Name AS AwayB3Name,
        al.B4Name AS AwayB4Name, al.B5Name AS AwayB5Name, al.B6Name AS AwayB6Name,
        al.B7Name AS AwayB7Name, al.B8Name AS AwayB8Name, al.B9Name AS AwayB9Name,
        al.B1IDFG AS AwayBatter1IDFG, al.B2IDFG AS AwayBatter2IDFG, al.B3IDFG AS AwayBatter3IDFG, al.B4IDFG AS AwayBatter4IDFG, al.B5IDFG as AwayBatter5IDFG,
        al.B6IDFG AS AwayBatter6IDFG, al.B7IDFG AS AwayBatter7IDFG, al.B8IDFG AS AwayBatter8IDFG, al.B9IDFG AS AwayBatter9IDFG,
        al.B1IDMLB AS AwayBatter1IDMLB, al.B2IDMLB AS AwayBatter2IDMLB, al.B3IDMLB AS AwayBatter3IDMLB, al.B4IDMLB AS AwayBatter4IDMLB, al.B5IDMLB AS AwayBatter5IDMLB,
        al.B6IDMLB AS AwayBatter6IDMLB, al.B7IDMLB AS AwayBatter7IDMLB, al.B8IDMLB AS AwayBatter8IDMLB, al.B9IDMLB AS AwayBatter9IDMLB,
        al.IDMLB_1B AS Away1BIDMLB, al.IDMLB_2B AS Away2BIDMLB, al.IDMLB_SS AS AwaySSIDMLB,
        al.IDMLB_3B AS Away3BIDMLB, al.IDMLB_C AS AwayCIDMLB, al.IDMLB_LF AS AwayLFIDMLB,
        al.IDMLB_CF AS AwayCFIDMLB, al.IDMLB_RF AS AwayRFIDMLB
        FROM gbs_base1 g
        LEFT JOIN lineups_wide hl ON g.HomeTeam = hl.Team
        LEFT JOIN lineups_wide al ON g.AwayTeam = al.Team
    """).df()
    
    master_wins = pd.read_csv('/Users/owendrummond/Documents/python_projects/Capstone/master_wins.csv')
    
    team_history = duckdb.query("""
    SELECT Date, AwayTeam as Team, AwayScore as Scored, HomeScore as Allowed, AwayTeamWin as Win FROM master_wins
    UNION ALL
    SELECT Date, HomeTeam as Team, HomeScore as Scored, AwayScore as Allowed, HomeTeamWin as Win FROM master_wins
    """).df()

    # Calculate only the requested windows
    latest_rolling = duckdb.query("""
        SELECT 
            Team,
            SUM(Win) OVER(PARTITION BY Team ORDER BY Date ROWS BETWEEN 162 PRECEDING AND 1 PRECEDING) as WinsL162,
            SUM(Win) OVER(PARTITION BY Team ORDER BY Date ROWS BETWEEN 50 PRECEDING AND 1 PRECEDING) as WinsL50,
            SUM(Scored) OVER(PARTITION BY Team ORDER BY Date ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING) as RunsScoredL100,
            SUM(Allowed) OVER(PARTITION BY Team ORDER BY Date ROWS BETWEEN 100 PRECEDING AND 1 PRECEDING) as RunsAllowedL100
        FROM team_history
        QUALIFY row_number() OVER(PARTITION BY Team ORDER BY Date DESC) = 1
    """).df()

    # 3. Join to gbs_pre_stats
    gbs_with_wins = duckdb.query("""
        SELECT 
            g.*,
            -- Away Team Features
            al.WinsL162 AS AwayTeamWinsL162,
            al.WinsL50 AS AwayTeamWinsL50,
            al.RunsScoredL100 AS AwayTeamRunsScoredL100,
            al.RunsAllowedL100 AS AwayTeamRunsAllowedL100,
            
            -- Home Team Features
            hl.WinsL162 AS HomeTeamWinsL162,
            hl.WinsL50 AS HomeTeamWinsL50,
            hl.RunsScoredL100 AS HomeTeamRunsScoredL100,
            hl.RunsAllowedL100 AS HomeTeamRunsAllowedL100
        FROM gbs_pre_stats g
        LEFT JOIN latest_rolling al ON g.AwayTeam = al.Team
        LEFT JOIN latest_rolling hl ON g.HomeTeam = hl.Team
    """).df()
    
    today_val = pd.to_datetime('today').date()
    year_val = int(today_val.year)

    gbs_with_wins.insert(0, 'Date', today_val)
    gbs_with_wins.insert(1, 'Year', year_val)

    gbs_with_wins['Date'] = pd.to_datetime(gbs_with_wins['Date'])
    
    GameBoxScoresSPG = duckdb.query("""
    with stats_weighted as (
        SELECT
            playerid,
            season,
            IP,
            Events,
            Date,
            ("K/BB" * IP) as K_BB_W,
            (xERA * IP) as xERA_W,
            ("Barrel%" * Events) as Barrel_W
        FROM SPGameLogFact
    )

    select GBS.*,
        hp.HomeSP_IP_L4A,
        hp.HomeSP_K_BB_L4A,
        hp.HomeSP_xERA_L4A,
        hp.HomeSP_Barrel_pct_L4A,

        ap.AwaySP_IP_L4A,
        ap.AwaySP_K_BB_L4A,
        ap.AwaySP_xERA_L4A,
        ap.AwaySP_Barrel_pct_L4A

    FROM gbs_with_wins GBS

    LEFT JOIN LATERAL(
        select 
        SUM(IP) as HomeSP_IP_L4A,
        SUM(K_BB_W) / NULLIF(SUM(IP), 0) as HomeSP_K_BB_L4A,
        SUM(xERA_W) / NULLIF(SUM(IP), 0) as HomeSP_xERA_L4A,
        SUM(Barrel_W) / NULLIF(SUM(Events), 0) as HomeSP_Barrel_pct_L4A
        FROM(
            SELECT
            IP,
            Events,
            K_BB_W,
            xERA_W,
            Barrel_W
            from stats_weighted sw
            where sw.playerid = GBS.HomePitcherIDFG
            and sw.season = GBS.Year
            and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC
            limit 4
        )
    ) hp on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(IP) as AwaySP_IP_L4A,
        SUM(K_BB_W) / NULLIF(SUM(IP), 0) as AwaySP_K_BB_L4A,
        SUM(xERA_W) / NULLIF(SUM(IP), 0) as AwaySP_xERA_L4A,
        SUM(Barrel_W) / NULLIF(SUM(Events), 0) as AwaySP_Barrel_pct_L4A
        FROM(
            SELECT
            IP,
            Events,
            K_BB_W,
            xERA_W,
            Barrel_W,
            from stats_weighted sw
            where sw.playerid = GBS.AwayPitcherIDFG
            and sw.season = GBS.Year
            and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC
            limit 4
        )
    ) ap on TRUE

    """).df()
    
    GameBoxScoresSPG = duckdb.query("""
    with stats_weighted as (
        SELECT
            playerid, season, AB, PA, Events, Date, 
            ("BB/K" * PA) as BB_K_W, 
            ("wRC+" * PA) as wRC_plus_W, 
            (xAVG * AB) as xAVG_W, 
            (xSLG * AB) as xSLG_W
        FROM BatterGameLogFact
    )

    select GBS.*,
        HomeB1_PA_L10A, HomeB1_BB_K_L10A, HomeB1_wRC_plus_L10A, HomeB1_xAVG_L10A, HomeB1_xSLG_L10A,
        HomeB2_PA_L10A, HomeB2_BB_K_L10A, HomeB2_wRC_plus_L10A, HomeB2_xAVG_L10A, HomeB2_xSLG_L10A,
        HomeB3_PA_L10A, HomeB3_BB_K_L10A, HomeB3_wRC_plus_L10A, HomeB3_xAVG_L10A, HomeB3_xSLG_L10A,
        HomeB4_PA_L10A, HomeB4_BB_K_L10A, HomeB4_wRC_plus_L10A, HomeB4_xAVG_L10A, HomeB4_xSLG_L10A,
        HomeB5_PA_L10A, HomeB5_BB_K_L10A, HomeB5_wRC_plus_L10A, HomeB5_xAVG_L10A, HomeB5_xSLG_L10A,
        HomeB6_PA_L10A, HomeB6_BB_K_L10A, HomeB6_wRC_plus_L10A, HomeB6_xAVG_L10A, HomeB6_xSLG_L10A,
        HomeB7_PA_L10A, HomeB7_BB_K_L10A, HomeB7_wRC_plus_L10A, HomeB7_xAVG_L10A, HomeB7_xSLG_L10A,
        HomeB8_PA_L10A, HomeB8_BB_K_L10A, HomeB8_wRC_plus_L10A, HomeB8_xAVG_L10A, HomeB8_xSLG_L10A,
        HomeB9_PA_L10A, HomeB9_BB_K_L10A, HomeB9_wRC_plus_L10A, HomeB9_xAVG_L10A, HomeB9_xSLG_L10A,
        
        AwayB1_PA_L10A, AwayB1_BB_K_L10A, AwayB1_wRC_plus_L10A, AwayB1_xAVG_L10A, AwayB1_xSLG_L10A,
        AwayB2_PA_L10A, AwayB2_BB_K_L10A, AwayB2_wRC_plus_L10A, AwayB2_xAVG_L10A, AwayB2_xSLG_L10A,
        AwayB3_PA_L10A, AwayB3_BB_K_L10A, AwayB3_wRC_plus_L10A, AwayB3_xAVG_L10A, AwayB3_xSLG_L10A,
        AwayB4_PA_L10A, AwayB4_BB_K_L10A, AwayB4_wRC_plus_L10A, AwayB4_xAVG_L10A, AwayB4_xSLG_L10A,
        AwayB5_PA_L10A, AwayB5_BB_K_L10A, AwayB5_wRC_plus_L10A, AwayB5_xAVG_L10A, AwayB5_xSLG_L10A,
        AwayB6_PA_L10A, AwayB6_BB_K_L10A, AwayB6_wRC_plus_L10A, AwayB6_xAVG_L10A, AwayB6_xSLG_L10A,
        AwayB7_PA_L10A, AwayB7_BB_K_L10A, AwayB7_wRC_plus_L10A, AwayB7_xAVG_L10A, AwayB7_xSLG_L10A,
        AwayB8_PA_L10A, AwayB8_BB_K_L10A, AwayB8_wRC_plus_L10A, AwayB8_xAVG_L10A, AwayB8_xSLG_L10A,
        AwayB9_PA_L10A, AwayB9_BB_K_L10A, AwayB9_wRC_plus_L10A, AwayB9_xAVG_L10A, AwayB9_xSLG_L10A

    FROM GameBoxScoresSPG GBS

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as HomeB1_PA_L10A,
            SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB1_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB1_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB1_xAVG_L10A,
            SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB1_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.HomeBatter1IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) hb1 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as HomeB2_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB2_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB2_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB2_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB2_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.HomeBatter2IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) hb2 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as HomeB3_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB3_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB3_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB3_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB3_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.HomeBatter3IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) hb3 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as HomeB4_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB4_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB4_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB4_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB4_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.HomeBatter4IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) hb4 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as HomeB5_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB5_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB5_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB5_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB5_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.HomeBatter5IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) hb5 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as HomeB6_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB6_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB6_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB6_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB6_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.HomeBatter6IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) hb6 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as HomeB7_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB7_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB7_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB7_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB7_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.HomeBatter7IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) hb7 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as HomeB8_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB8_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB8_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB8_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB8_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.HomeBatter8IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) hb8 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as HomeB9_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as HomeB9_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB9_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as HomeB9_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as HomeB9_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.HomeBatter9IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) hb9 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as AwayB1_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB1_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB1_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB1_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB1_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.AwayBatter1IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) ab1 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as AwayB2_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB2_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB2_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB2_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB2_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.AwayBatter2IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) ab2 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as AwayB3_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB3_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB3_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB3_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB3_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.AwayBatter3IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) ab3 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as AwayB4_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB4_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB4_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB4_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB4_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.AwayBatter4IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) ab4 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as AwayB5_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB5_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB5_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB5_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB5_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.AwayBatter5IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) ab5 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as AwayB6_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB6_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB6_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB6_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB6_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.AwayBatter6IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) ab6 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as AwayB7_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB7_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB7_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB7_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB7_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.AwayBatter7IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) ab7 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as AwayB8_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB8_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB8_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB8_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB8_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.AwayBatter8IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) ab8 on TRUE

    LEFT JOIN LATERAL(
        select 
            SUM(PA) as AwayB9_PA_L10A, SUM(BB_K_W) / NULLIF(SUM(PA), 0) as AwayB9_BB_K_L10A,
            SUM(wRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB9_wRC_plus_L10A,
            SUM(xAVG_W) / NULLIF(SUM(AB), 0) as AwayB9_xAVG_L10A, SUM(xSLG_W) / NULLIF(SUM(AB), 0) as AwayB9_xSLG_L10A
        FROM(
            SELECT AB, PA, BB_K_W, wRC_plus_W, xAVG_W, xSLG_W
            from stats_weighted sw
            where sw.playerid = GBS.AwayBatter9IDFG and sw.season = GBS.Year and cast(sw.Date as date) < cast(GBS.Date as date)
            order by sw.Date DESC limit 10
        )
    ) ab9 on TRUE

    """).df()
    
    GameBoxScoresSPG = duckdb.query("""
    with stats_weighted as (
            SELECT
            bpid,
            mlbid,
            Year,
            Month,
            IP,
            CompDate,
            (DRA * IP) as DRA_W,
            ("K%" * IP) as Kpct_W,
            ("BB%" * IP) as BBpct_W
        FROM dra_map_clean
    )


    select GBS.*,
    hp.HomeSP_IP_L5M,
    hp.HomeSP_DRA_L5M,
    hp.HomeSP_Kpct_L5M,
    hp.HomeSP_BBpct_L5M,
    ap.AwaySP_IP_L5M,
    ap.AwaySP_DRA_L5M,
    ap.AwaySP_Kpct_L5M,
    ap.AwaySP_BBpct_L5M

    FROM GameBoxScoresSPG GBS

    LEFT JOIN LATERAL(
        select 
        SUM(IP) as HomeSP_IP_L5M,
        SUM(DRA_W) / NULLIF(SUM(IP), 0) as HomeSP_DRA_L5M,
        SUM(Kpct_W) / NULLIF(SUM(IP), 0) as HomeSP_Kpct_L5M,
        SUM(BBpct_W) / NULLIF(SUM(IP), 0) as HomeSP_BBpct_L5M
        FROM(
            SELECT
            IP,
            DRA_W,
            Kpct_W,
            BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomePitcherIDMLB
            and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC
            limit 5
        )
    ) hp on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(IP) as AwaySP_IP_L5M,
        SUM(DRA_W) / NULLIF(SUM(IP), 0) as AwaySP_DRA_L5M,
        SUM(Kpct_W) / NULLIF(SUM(IP), 0) as AwaySP_Kpct_L5M,
        SUM(BBpct_W) / NULLIF(SUM(IP), 0) as AwaySP_BBpct_L5M
        FROM(
            SELECT
            IP,
            DRA_W,
            Kpct_W,
            BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayPitcherIDMLB
            and CAST(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC
            limit 5
        )
    ) ap on TRUE

    """).df()
    
    GameBoxScores_bphitting = duckdb.query("""
    with stats_weighted as (
            SELECT bpid, mlbid, Year, Month, PA, CompDate, (OPS * PA) as OPS_W, (DRC_plus * PA) as DRC_plus_W,
            ("K%" * PA) as Kpct_W
        FROM drc_map_clean
    )

    select GBS.*,

    hb1.HomeB1_PA_L5M,hb1.HomeB1_OPS_L5M,hb1.HomeB1_DRC_plus_L5M,hb1.HomeB1_Kpct_L5M,
    hb2.HomeB2_PA_L5M,hb2.HomeB2_OPS_L5M,hb2.HomeB2_DRC_plus_L5M,hb2.HomeB2_Kpct_L5M,
    hb3.HomeB3_PA_L5M,hb3.HomeB3_OPS_L5M,hb3.HomeB3_DRC_plus_L5M,hb3.HomeB3_Kpct_L5M,
    hb4.HomeB4_PA_L5M,hb4.HomeB4_OPS_L5M,hb4.HomeB4_DRC_plus_L5M,hb4.HomeB4_Kpct_L5M,
    hb5.HomeB5_PA_L5M,hb5.HomeB5_OPS_L5M,hb5.HomeB5_DRC_plus_L5M,hb5.HomeB5_Kpct_L5M,
    hb6.HomeB6_PA_L5M,hb6.HomeB6_OPS_L5M,hb6.HomeB6_DRC_plus_L5M,hb6.HomeB6_Kpct_L5M,
    hb7.HomeB7_PA_L5M,hb7.HomeB7_OPS_L5M,hb7.HomeB7_DRC_plus_L5M,hb7.HomeB7_Kpct_L5M,
    hb8.HomeB8_PA_L5M,hb8.HomeB8_OPS_L5M,hb8.HomeB8_DRC_plus_L5M,hb8.HomeB8_Kpct_L5M,
    hb9.HomeB9_PA_L5M,hb9.HomeB9_OPS_L5M,hb9.HomeB9_DRC_plus_L5M,hb9.HomeB9_Kpct_L5M,

    ab1.AwayB1_PA_L5M,ab1.AwayB1_OPS_L5M,ab1.AwayB1_DRC_plus_L5M,ab1.AwayB1_Kpct_L5M,
    ab2.AwayB2_PA_L5M,ab2.AwayB2_OPS_L5M,ab2.AwayB2_DRC_plus_L5M,ab2.AwayB2_Kpct_L5M,
    ab3.AwayB3_PA_L5M,ab3.AwayB3_OPS_L5M,ab3.AwayB3_DRC_plus_L5M,ab3.AwayB3_Kpct_L5M,
    ab4.AwayB4_PA_L5M,ab4.AwayB4_OPS_L5M,ab4.AwayB4_DRC_plus_L5M,ab4.AwayB4_Kpct_L5M,
    ab5.AwayB5_PA_L5M,ab5.AwayB5_OPS_L5M,ab5.AwayB5_DRC_plus_L5M,ab5.AwayB5_Kpct_L5M,
    ab6.AwayB6_PA_L5M,ab6.AwayB6_OPS_L5M,ab6.AwayB6_DRC_plus_L5M,ab6.AwayB6_Kpct_L5M,
    ab7.AwayB7_PA_L5M,ab7.AwayB7_OPS_L5M,ab7.AwayB7_DRC_plus_L5M,ab7.AwayB7_Kpct_L5M,
    ab8.AwayB8_PA_L5M,ab8.AwayB8_OPS_L5M,ab8.AwayB8_DRC_plus_L5M,ab8.AwayB8_Kpct_L5M,
    ab9.AwayB9_PA_L5M,ab9.AwayB9_OPS_L5M,ab9.AwayB9_DRC_plus_L5M,ab9.AwayB9_Kpct_L5M

    FROM GameBoxScoresSPG GBS

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as HomeB1_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB1_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB1_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB1_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter1IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) hb1 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as HomeB2_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB2_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB2_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB2_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter2IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) hb2 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as HomeB3_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB3_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB3_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB3_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter3IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) hb3 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as HomeB4_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB4_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB4_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB4_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter4IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) hb4 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as HomeB5_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB5_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB5_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB5_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter5IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) hb5 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as HomeB6_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB6_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB6_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB6_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter6IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) hb6 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as HomeB7_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB7_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB7_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB7_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter7IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) hb7 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as HomeB8_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB8_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB8_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB8_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter8IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) hb8 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as HomeB9_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB9_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as HomeB9_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as HomeB9_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter9IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) hb9 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as AwayB1_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB1_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB1_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB1_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter1IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) ab1 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as AwayB2_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB2_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB2_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB2_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter2IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) ab2 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as AwayB3_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB3_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB3_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB3_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter3IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) ab3 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as AwayB4_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB4_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB4_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB4_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter4IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) ab4 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as AwayB5_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB5_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB5_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB5_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter5IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) ab5 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as AwayB6_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB6_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB6_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB6_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter6IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) ab6 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as AwayB7_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB7_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB7_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB7_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter7IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) ab7 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as AwayB8_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB8_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB8_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB8_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter8IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) ab8 on TRUE

    LEFT JOIN LATERAL (
        select 
        SUM(PA) as AwayB9_PA_L5M, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB9_OPS_L5M,
        SUM(DRC_plus_W) / NULLIF(SUM(PA), 0) as AwayB9_DRC_plus_L5M, SUM(Kpct_W) / NULLIF(SUM(PA), 0) as AwayB9_Kpct_L5M
        FROM (
            SELECT PA,OPS_W,DRC_plus_W,Kpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter9IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC limit 5
        )
    ) ab9 on TRUE

    """).df()
    
    GameBoxScores_phitting = duckdb.query("""
    with stats_weighted as (
            SELECT bpid, mlbid, Year, Month, PlatoonHand, PA, CompDate, (OPS * PA) as OPS_W,
            ("BB%" * PA) as BBpct_W
        FROM drcp_map_clean
    )

    select GBS.*,

    hb1.HomeB1_PA_L5M_P,hb1.HomeB1_OPS_L5M_P,hb1.HomeB1_BBpct_L5M_P,
    hb2.HomeB2_PA_L5M_P,hb2.HomeB2_OPS_L5M_P,hb2.HomeB2_BBpct_L5M_P,
    hb3.HomeB3_PA_L5M_P,hb3.HomeB3_OPS_L5M_P,hb3.HomeB3_BBpct_L5M_P,
    hb4.HomeB4_PA_L5M_P,hb4.HomeB4_OPS_L5M_P,hb4.HomeB4_BBpct_L5M_P,
    hb5.HomeB5_PA_L5M_P,hb5.HomeB5_OPS_L5M_P,hb5.HomeB5_BBpct_L5M_P,
    hb6.HomeB6_PA_L5M_P,hb6.HomeB6_OPS_L5M_P,hb6.HomeB6_BBpct_L5M_P,
    hb7.HomeB7_PA_L5M_P,hb7.HomeB7_OPS_L5M_P,hb7.HomeB7_BBpct_L5M_P,
    hb8.HomeB8_PA_L5M_P,hb8.HomeB8_OPS_L5M_P,hb8.HomeB8_BBpct_L5M_P,
    hb9.HomeB9_PA_L5M_P,hb9.HomeB9_OPS_L5M_P,hb9.HomeB9_BBpct_L5M_P,

    ab1.AwayB1_PA_L5M_P,ab1.AwayB1_OPS_L5M_P,ab1.AwayB1_BBpct_L5M_P,
    ab2.AwayB2_PA_L5M_P,ab2.AwayB2_OPS_L5M_P,ab2.AwayB2_BBpct_L5M_P,
    ab3.AwayB3_PA_L5M_P,ab3.AwayB3_OPS_L5M_P,ab3.AwayB3_BBpct_L5M_P,
    ab4.AwayB4_PA_L5M_P,ab4.AwayB4_OPS_L5M_P,ab4.AwayB4_BBpct_L5M_P,
    ab5.AwayB5_PA_L5M_P,ab5.AwayB5_OPS_L5M_P,ab5.AwayB5_BBpct_L5M_P,
    ab6.AwayB6_PA_L5M_P,ab6.AwayB6_OPS_L5M_P,ab6.AwayB6_BBpct_L5M_P,
    ab7.AwayB7_PA_L5M_P,ab7.AwayB7_OPS_L5M_P,ab7.AwayB7_BBpct_L5M_P,
    ab8.AwayB8_PA_L5M_P,ab8.AwayB8_OPS_L5M_P,ab8.AwayB8_BBpct_L5M_P,
    ab9.AwayB9_PA_L5M_P,ab9.AwayB9_OPS_L5M_P,ab9.AwayB9_BBpct_L5M_P

    FROM GameBoxScores_bphitting GBS

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as HomeB1_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB1_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB1_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter1IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) hb1 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as HomeB2_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB2_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB2_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter2IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) hb2 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as HomeB3_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB3_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB3_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter3IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) hb3 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as HomeB4_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB4_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB4_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter4IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) hb4 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as HomeB5_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB5_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB5_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter5IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) hb5 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as HomeB6_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB6_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB6_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter6IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) hb6 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as HomeB7_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB7_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB7_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter7IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) hb7 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as HomeB8_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB8_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB8_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter8IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) hb8 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as HomeB9_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as HomeB9_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as HomeB9_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.HomeBatter9IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = AwayPitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) hb9 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as AwayB1_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB1_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB1_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter1IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) ab1 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as AwayB2_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB2_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB2_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter2IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) ab2 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as AwayB3_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB3_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB3_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter3IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) ab3 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as AwayB4_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB4_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB4_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter4IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) ab4 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as AwayB5_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB5_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB5_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter5IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) ab5 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as AwayB6_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB6_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB6_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter6IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) ab6 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as AwayB7_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB7_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB7_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter7IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) ab7 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as AwayB8_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB8_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB8_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter8IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) ab8 on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(PA) as AwayB9_PA_L5M_P, SUM(OPS_W) / NULLIF(SUM(PA), 0) as AwayB9_OPS_L5M_P,
        SUM(BBpct_W) / NULLIF(SUM(PA), 0) as AwayB9_BBpct_L5M_P
        FROM(
            SELECT
            PA,OPS_W,BBpct_W
            from stats_weighted sw
            where sw.mlbid = GBS.AwayBatter9IDMLB and cast(sw.CompDate as date) < cast(GBS.Date as date) and sw.PlatoonHand = HomePitcherHand
            order by sw.CompDate DESC
            limit 5
        )
    ) ab9 on TRUE

    """).df()
        
    
    GameBoxScores_w_bullpen = duckdb.query("""
    with stats_weighted as (
            SELECT
            Team_Abbr,
            query_year,
            query_month,
            IP,
            CompDate,
            (ERA * IP) as ERA_W,
            (SO9 * IP) as SO9_W,
            (BB9 * IP) as BB9_W
        FROM team_bullpen_monthly
    )


    select GBS.*,
    hp.HomeBP_ERA_L3M,
    hp.HomeBP_SO9_L3M,
    hp.HomeBP_BB9_L3M,
    ap.AwayBP_ERA_L3M,
    ap.AwayBP_SO9_L3M,
    ap.AwayBP_BB9_L3M

    FROM GameBoxScores_phitting GBS

    LEFT JOIN LATERAL(
        select 
        SUM(ERA_W) / NULLIF(SUM(IP), 0) as HomeBP_ERA_L3M,
        SUM(SO9_W) / NULLIF(SUM(IP), 0) as HomeBP_SO9_L3M,
        SUM(BB9_W) / NULLIF(SUM(IP), 0) as HomeBP_BB9_L3M
        FROM(
            SELECT
            IP,
            ERA_W,
            SO9_W,
            BB9_W
            from stats_weighted sw
            where sw.Team_Abbr = GBS.HomeTeam
            and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC
            limit 3
        )
    ) hp on TRUE

    LEFT JOIN LATERAL(
        select 
        SUM(ERA_W) / NULLIF(SUM(IP), 0) as AwayBP_ERA_L3M,
        SUM(SO9_W) / NULLIF(SUM(IP), 0) as AwayBP_SO9_L3M,
        SUM(BB9_W) / NULLIF(SUM(IP), 0) as AwayBP_BB9_L3M
        FROM(
            SELECT
            IP,
            ERA_W,
            SO9_W,
            BB9_W
            from stats_weighted sw
            where sw.Team_Abbr = GBS.AwayTeam
            and cast(sw.CompDate as date) < cast(GBS.Date as date)
            order by sw.CompDate DESC
            limit 3
        )
    ) ap on TRUE

    """).df()
    
    GameBoxScores_fghitting = duckdb.query("""
    WITH stats_weighted AS (
        SELECT
            xMLBAMID, Season, AB, PA, Events, 
            BaseRunning,
            (BB_K * PA) as BB_K_W,
            (wRC_plus * PA) as wRC_plus_W,
            (xwOBA * PA) as xwOBA_W,
            (LA * Events) as LA_W
        FROM fangraphs_hitting_fact
    )

    SELECT GBS.*,
        hb1.HomeB1_BSR_L2, hb1.HomeB1_BB_K_L2, hb1.HomeB1_wRC_plus_L2, hb1.HomeB1_xwOBA_L2, hb1.HomeB1_LA_L2,
        hb2.HomeB2_BSR_L2, hb2.HomeB2_BB_K_L2, hb2.HomeB2_wRC_plus_L2, hb2.HomeB2_xwOBA_L2, hb2.HomeB2_LA_L2,
        hb3.HomeB3_BSR_L2, hb3.HomeB3_BB_K_L2, hb3.HomeB3_wRC_plus_L2, hb3.HomeB3_xwOBA_L2, hb3.HomeB3_LA_L2,
        hb4.HomeB4_BSR_L2, hb4.HomeB4_BB_K_L2, hb4.HomeB4_wRC_plus_L2, hb4.HomeB4_xwOBA_L2, hb4.HomeB4_LA_L2,
        hb5.HomeB5_BSR_L2, hb5.HomeB5_BB_K_L2, hb5.HomeB5_wRC_plus_L2, hb5.HomeB5_xwOBA_L2, hb5.HomeB5_LA_L2,
        hb6.HomeB6_BSR_L2, hb6.HomeB6_BB_K_L2, hb6.HomeB6_wRC_plus_L2, hb6.HomeB6_xwOBA_L2, hb6.HomeB6_LA_L2,
        hb7.HomeB7_BSR_L2, hb7.HomeB7_BB_K_L2, hb7.HomeB7_wRC_plus_L2, hb7.HomeB7_xwOBA_L2, hb7.HomeB7_LA_L2,
        hb8.HomeB8_BSR_L2, hb8.HomeB8_BB_K_L2, hb8.HomeB8_wRC_plus_L2, hb8.HomeB8_xwOBA_L2, hb8.HomeB8_LA_L2,
        hb9.HomeB9_BSR_L2, hb9.HomeB9_BB_K_L2, hb9.HomeB9_wRC_plus_L2, hb9.HomeB9_xwOBA_L2, hb9.HomeB9_LA_L2,
        ab1.AwayB1_BSR_L2, ab1.AwayB1_BB_K_L2, ab1.AwayB1_wRC_plus_L2, ab1.AwayB1_xwOBA_L2, ab1.AwayB1_LA_L2,
        ab2.AwayB2_BSR_L2, ab2.AwayB2_BB_K_L2, ab2.AwayB2_wRC_plus_L2, ab2.AwayB2_xwOBA_L2, ab2.AwayB2_LA_L2,
        ab3.AwayB3_BSR_L2, ab3.AwayB3_BB_K_L2, ab3.AwayB3_wRC_plus_L2, ab3.AwayB3_xwOBA_L2, ab3.AwayB3_LA_L2,
        ab4.AwayB4_BSR_L2, ab4.AwayB4_BB_K_L2, ab4.AwayB4_wRC_plus_L2, ab4.AwayB4_xwOBA_L2, ab4.AwayB4_LA_L2,
        ab5.AwayB5_BSR_L2, ab5.AwayB5_BB_K_L2, ab5.AwayB5_wRC_plus_L2, ab5.AwayB5_xwOBA_L2, ab5.AwayB5_LA_L2,
        ab6.AwayB6_BSR_L2, ab6.AwayB6_BB_K_L2, ab6.AwayB6_wRC_plus_L2, ab6.AwayB6_xwOBA_L2, ab6.AwayB6_LA_L2,
        ab7.AwayB7_BSR_L2, ab7.AwayB7_BB_K_L2, ab7.AwayB7_wRC_plus_L2, ab7.AwayB7_xwOBA_L2, ab7.AwayB7_LA_L2,
        ab8.AwayB8_BSR_L2, ab8.AwayB8_BB_K_L2, ab8.AwayB8_wRC_plus_L2, ab8.AwayB8_xwOBA_L2, ab8.AwayB8_LA_L2,
        ab9.AwayB9_BSR_L2, ab9.AwayB9_BB_K_L2, ab9.AwayB9_wRC_plus_L2, ab9.AwayB9_xwOBA_L2, ab9.AwayB9_LA_L2

    FROM GameBoxScores_w_bullpen GBS

    -- HOME BATTERS 1-9
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB1_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB1_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB1_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB1_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB1_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter1IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb1 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB2_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB2_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB2_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB2_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB2_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter2IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb2 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB3_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB3_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB3_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB3_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB3_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter3IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb3 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB4_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB4_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB4_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB4_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB4_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter4IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb4 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB5_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB5_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB5_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB5_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB5_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter5IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb5 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB6_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB6_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB6_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB6_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB6_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter6IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb6 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB7_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB7_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB7_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB7_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB7_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter7IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb7 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB8_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB8_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB8_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB8_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB8_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter8IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb8 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as HomeB9_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as HomeB9_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as HomeB9_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as HomeB9_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as HomeB9_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.HomeBatter9IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) hb9 ON TRUE

    -- AWAY BATTERS 1-9
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB1_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB1_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB1_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB1_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB1_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter1IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab1 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB2_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB2_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB2_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB2_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB2_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter2IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab2 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB3_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB3_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB3_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB3_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB3_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter3IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab3 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB4_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB4_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB4_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB4_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB4_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter4IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab4 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB5_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB5_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB5_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB5_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB5_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter5IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab5 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB6_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB6_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB6_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB6_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB6_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter6IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab6 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB7_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB7_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB7_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB7_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB7_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter7IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab7 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB8_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB8_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB8_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB8_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB8_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter8IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab8 ON TRUE
    LEFT JOIN LATERAL (SELECT SUM(BaseRunning) as AwayB9_BSR_L2, SUM(BB_K_W)/NULLIF(SUM(PA),0) as AwayB9_BB_K_L2, SUM(wRC_plus_W)/NULLIF(SUM(PA),0) as AwayB9_wRC_plus_L2, SUM(xwOBA_W)/NULLIF(SUM(PA),0) as AwayB9_xwOBA_L2, SUM(LA_W)/NULLIF(SUM(Events),0) as AwayB9_LA_L2 FROM (SELECT PA, Events, BaseRunning, BB_K_W, wRC_plus_W, xwOBA_W, LA_W FROM stats_weighted sw WHERE sw.xMLBAMID = GBS.AwayBatter9IDMLB AND sw.Season < GBS.Year ORDER BY sw.Season DESC LIMIT 2)) ab9 ON TRUE

    """).df()
    
    GameBoxScores_fgpitching = duckdb.query("""
    WITH stats_weighted AS (
        SELECT
            xMLBAMID,
            Season,
            IP,
            Events,
            WAR,
            (ERA * IP) as ERA_W,
            (K_BB * IP) as K_BB_W,
            (xFIP * IP) as xFIP_W,
            (SIERA * IP) as SIERA_W,
            (GB_pct * Events) as GB_pct_W
        FROM fangraphs_pitching_fact
    )

    select GBS.*,
        -- Home Pitcher L2 Stats
        hP.HomeSP_WAR_L2,
        hP.HomeSP_ERA_L2,
        hP.HomeSP_K_BB_L2,
        hP.HomeSP_xFIP_L2,
        hP.HomeSP_SIERA_L2,
        hp.HomeSP_GB_pct_L2,
        
        -- Away Pitcher L2 Stats
    ap.AwaySP_WAR_L2,
    ap.AwaySP_ERA_L2,
    ap.AwaySP_K_BB_L2,
    ap.AwaySP_xFIP_L2,
    ap.AwaySP_SIERA_L2,
    ap.AwaySP_GB_pct_L2

    FROM GameBoxScores_fghitting GBS

    LEFT JOIN LATERAL(
        select 
        SUM(IP) as HomeSP_IP_L2,
        SUM(WAR) as HomeSP_WAR_L2,
        SUM(ERA_W) / NULLIF(SUM(IP), 0) as HomeSP_ERA_L2,
        SUM(K_BB_W) / NULLIF(SUM(IP), 0) as HomeSP_K_BB_L2,
        SUM(xFIP_W) / NULLIF(SUM(IP), 0) as HomeSP_xFIP_L2,
        SUM(SIERA_W) / NULLIF(SUM(IP), 0) as HomeSP_SIERA_L2,
        SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as HomeSP_GB_pct_L2
        FROM(
            SELECT
            IP,
            Events,
            WAR,
            ERA_W,
            K_BB_W,
            xFIP_W,
            SIERA_W,
            GB_pct_W
            from stats_weighted sw
            where sw.xMLBAMID = GBS.HomePitcherIDMLB
            and sw.Season < GBS.Year
            order by sw.Season DESC
            limit 2
        )
    ) hp on TRUE

    LEFT JOIN LATERAL(
    select
    SUM(IP) as AwaySP_IP_L2,
    SUM(WAR) as AwaySP_WAR_L2,
    SUM(ERA_W) / NULLIF(SUM(IP), 0) as AwaySP_ERA_L2,
    SUM(K_BB_W) / NULLIF(SUM(IP), 0) as AwaySP_K_BB_L2,
    SUM(xFIP_W) / NULLIF(SUM(IP), 0) as AwaySP_xFIP_L2,
    SUM(SIERA_W) / NULLIF(SUM(IP), 0) as AwaySP_SIERA_L2,
    SUM(GB_pct_W) / NULLIF(SUM(Events), 0) as AwaySP_GB_pct_L2
    FROM(
        SELECT
        IP,
        Events,
        WAR,
        ERA_W,
        K_BB_W,
        xFIP_W,
        SIERA_W,
        GB_pct_W
        from stats_weighted sw
        where sw.xMLBAMID = GBS.AwayPitcherIDMLB
        and sw.Season < GBS.Year
        order by sw.Season DESC
        limit 2
    )
    ) ap on TRUE


    """).df()
    
    new_cols = ['DeltaFirstYearCount', 'HomePitcherIDFirstYear', 'AwayPitcherIDFirstYear']

    for col in new_cols:
        GameBoxScores_fghitting[col] = 0
        
    GameBoxScores = duckdb.query("""
    SELECT
    GBS.*,
    AC.AC_RUNS AS AwayFieldingRunsC,
    A1B.A1B_RUNS AS AwayFieldingRuns1B,
    A2B.A2B_RUNS AS AwayFieldingRuns2B,
    A3B.A3B_RUNS AS AwayFieldingRuns3B,
    ASS.ASS_RUNS AS AwayFieldingRunsSS,
    ALF.ALF_RUNS AS AwayFieldingRunsLF,
    ACF.ACF_RUNS AS AwayFieldingRunsCF,
    ARF.ARF_RUNS AS AwayFieldingRunsRF,
    HC.HC_RUNS AS HomeFieldingRunsC,
    H1B.H1B_RUNS AS HomeFieldingRuns1B,
    H2B.H2B_RUNS AS HomeFieldingRuns2B,
    H3B.H3B_RUNS AS HomeFieldingRuns3B,
    HSS.HSS_RUNS AS HomeFieldingRunsSS,
    HLF.HLF_RUNS AS HomeFieldingRunsLF,
    HCF.HCF_RUNS AS HomeFieldingRunsCF,
    HRF.HRF_RUNS AS HomeFieldingRunsRF


    FROM GameBoxScores_fgpitching GBS


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as AC_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.AwayCIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) AC ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as A1B_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.Away1BIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) A1B ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as A2B_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.Away2BIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) A2B ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as A3B_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.Away3BIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) A3B ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as ASS_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.AwaySSIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) ASS ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as ALF_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.AwayLFIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) ALF ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as ACF_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.AwayCFIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) ACF ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as ARF_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.AwayRFIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) ARF ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as HC_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.HomeCIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) HC ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as H1B_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.Home1BIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) H1B ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as H2B_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.Home2BIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) H2B ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as H3B_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.Home3BIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) H3B ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as HSS_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.HomeSSIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) HSS ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as HLF_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.HomeLFIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) HLF ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as HCF_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.HomeCFIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) HCF ON TRUE


    LEFT JOIN LATERAL(
    SELECT SUM(fielding_runs) as HRF_RUNS
    FROM(
        SELECT fielding_runs
        FROM defense_fact df
        WHERE df.mlb_id = GBS.HomeRFIDMLB
        AND CAST(df.CompDate as DATE) < CAST(GBS.Date as DATE)
        ORDER BY CAST(df.CompDate as DATE) DESC
        LIMIT 5
    )
    ) HRF ON TRUE
    """).df()
    
    GameBoxScores = clean1(GameBoxScores)
    #return(GameBoxScores)
    GameBoxScores = clean2(GameBoxScores)
    GameBoxScores = predict_records(GameBoxScores)
    
    return(GameBoxScores)


    

In [94]:
April252026 = daily_run()
April252026.to_csv('April252026.csv', index = False)

/var/folders/4k/3pfkn1jn039bp8wy8410pn3m0000gn/T/ipykernel_62423/2664521345.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  GameBoxScores[delta_col] = GameBoxScores[col] - GameBoxScores[away_col]
/var/folders/4k/3pfkn1jn039bp8wy8410pn3m0000gn/T/ipykernel_62423/2664521345.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  GameBoxScores[delta_col] = GameBoxScores[col] - GameBoxScores[away_col]
/var/folders/4k/3pfkn1jn039bp8wy8410pn3m0000gn/T/ipykernel_62423/2664521345.py:42: PerformanceWarning: DataFrame is highly fragmente

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 147ms/step


In [95]:
April252026

,Date,Year,HomeTeam,AwayTeam,HomePitcherIDFirstYear,AwayPitcherIDFirstYear,DeltaFirstYearCount,DeltaFieldingRunsC,DeltaSP_WAR_L2,DeltaSP_ERA_L2,...,Delta_Top_OPS_L5M_P_Avg,Delta_Middle_OPS_L5M_P_Avg,Delta_Bottom_OPS_L5M_P_Avg,Delta_Top_BBpct_L5M_P_Avg,Delta_Middle_BBpct_L5M_P_Avg,Delta_Bottom_BBpct_L5M_P_Avg,Delta_Infield_Defense_L5M,Delta_Outfield_Defense_L5M,final_prediction,inverse_prediction
0,2026-04-24,2026,MIL,PIT,0,0,0,-2.759463,-7.568000,0.769501,...,-0.015288,-0.033240,-0.009404,3.394872,3.347977,-1.240283,10.604256,11.401316,0.526687,0.473313
1,2026-04-24,2026,BAL,BOS,0,0,0,-4.204784,-3.969900,2.330938,...,-0.040484,-0.020840,-0.052554,4.140957,2.995402,-0.301244,-7.490823,-34.682814,0.447863,0.552137
2,2026-04-24,2026,STL,SEA,0,0,0,-6.985609,-4.039200,0.854094,...,-0.053318,-0.134003,-0.126871,-2.458511,-0.505590,-3.788768,21.288056,-1.155299,0.435103,0.564897
3,2026-04-24,2026,TOR,CLE,0,0,0,8.120576,-1.693000,1.149171,...,-0.024578,-0.033525,0.059440,-1.609455,-5.961573,0.363408,0.311877,16.026149,0.502833,0.497167
4,2026-04-24,2026,HOU,NYY,0,0,0,-3.778693,-1.559400,-0.613320,...,-0.068403,-0.123117,0.072712,-2.711998,-2.168078,-7.277599,-17.664207,-11.482149,0.431920,0.568080
5,2026-04-24,2026,ATL,PHI,0,0,0,0.244656,-1.723508,-0.118753,...,0.017141,-0.048974,-0.015418,1.955663,0.391321,0.327634,1.185358,-4.651960,0.514762,0.485238
6,2026-04-24,2026,NYM,COL,0,0,0,-7.800614,3.972700,-0.818122,...,0.030252,-0.012356,0.077753,8.695469,-1.255086,1.303169,-0.269194,-7.484922,0.611467,0.388533
7,2026-04-24,2026,CIN,DET,0,0,0,-14.623703,-2.721500,-0.045637,...,-0.245780,-0.055008,-0.287334,-5.771619,-3.255247,-0.099074,22.082783,3.875452,0.483321,0.516679
8,2026-04-24,2026,LAD,CHC,0,0,0,-9.674988,-0.845700,0.323113,...,0.153183,0.028030,-0.045698,5.326684,1.936432,-0.731599,-16.853775,-6.882984,0.560268,0.439732
9,2026-04-24,2026,TEX,ATH,0,0,0,3.675690,1.711800,-1.300490,...,-0.190714,-0.095875,-0.012814,2.517163,-3.138233,3.554097,14.386248,-2.547539,0.563404,0.436596
